In [1]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
from yellowbrick.cluster import KElbowVisualizer
import matplotlib.pyplot as plt
import pandas as pd 
import seaborn as sns
from sksurv.base import SurvivalAnalysisMixin as s
from sklearn.model_selection import train_test_split, RandomizedSearchCV, cross_val_score
from sksurv.preprocessing import encode_categorical
from sksurv.datasets import load_gbsg2
from sksurv.functions import StepFunction
from sksurv.linear_model import CoxPHSurvivalAnalysis, CoxnetSurvivalAnalysis
from sksurv.ensemble import (ComponentwiseGradientBoostingSurvivalAnalysis, 
                            RandomSurvivalForest, 
                            ExtraSurvivalTrees, 
                            GradientBoostingSurvivalAnalysis, 
                            ExtraSurvivalTrees)
from sksurv.meta import EnsembleSelection, EnsembleSelectionRegressor
from sksurv.metrics import integrated_brier_score
from matplotlib.colors import ListedColormap
from mlxtend.evaluate import paired_ttest_5x2cv
from mlxtend.evaluate import combined_ftest_5x2cv
from lifelines import KaplanMeierFitter
from scipy.cluster import hierarchy
from lifelines.statistics import logrank_test, multivariate_logrank_test, pairwise_logrank_test
from sklearn import preprocessing
from sklearn.model_selection import StratifiedKFold, KFold
from lifelines.plotting import add_at_risk_counts
import scipy.stats
import sklearn
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import optuna
from sklearn.model_selection import cross_val_score
from sksurv.metrics import integrated_brier_score
from lifelines import CoxPHFitter
from lifelines.statistics import proportional_hazard_test
import scipy.stats as stats
from statsmodels.stats.outliers_influence import variance_inflation_factor 
import statsmodels.api as sm
from IPython.core.interactiveshell import InteractiveShell
InteractiveShell.ast_node_interactivity = 'all'
from sklearn.preprocessing import StandardScaler

In [2]:
# OUS: Train data
OUS_D1 = pd.read_csv('OUS_D1.csv')
OUS_D2 = pd.read_csv('OUS_D2.csv')
OUS_D3 = pd.read_csv('OUS_D3.csv')
OUS_DFS_target = pd.read_csv('OUS_DFS_target.csv')
OUS_OS_target = pd.read_csv('OUS_OS_target.csv')
response_OUS = pd.read_csv('response_ous.csv', sep=';')

# MAASTRO: Test data 
MAASTRO_D1 = pd.read_csv('MAASTRO_D1.csv')
MAASTRO_D2 = pd.read_csv('MAASTRO_D2.csv')
MAASTRO_D3 = pd.read_csv('MAASTRO_D3.csv')
MAASTRO_DFS_target = pd.read_csv('MAASTRO_DFS_target.csv')
MAASTRO_OS_target = pd.read_csv('MAASTRO_OS_target.csv')
response_MAASTRO = pd.read_csv('maastro_response_full.csv', sep=',')

In [3]:
# Need to choose patient_id from OUS_D3 in response_OUS
data = list(OUS_D3['patient_id'])
mask = response_OUS['patient_id'].isin(data)
response_OUS = response_OUS[mask] 

# Merge OUS_D3 with response_OUS
clinical_train = pd.merge(OUS_D3, response_OUS, on='patient_id', how='inner')
clinical_train = clinical_train.loc[:, ~clinical_train.columns.isin(['OS', 'event_OS', 'LRC', 'event_LRC'])]

In [4]:
# Drop patient_id column
clinical_train = clinical_train.drop('patient_id', axis=1)

In [5]:
# Check null values in D3 
clinical_train.isnull().sum().sum()

0

### COX assumption in Train data

In [6]:
df = pd.DataFrame(clinical_train)

# Fit Cox proportional hazards model
cph = CoxPHFitter(penalizer=0.00001)
cph.fit(df, duration_col='DFS', event_col='event_DFS')

# Perform the proportional hazards assumption test
results = proportional_hazard_test(cph, df)
print(results)

<lifelines.CoxPHFitter: fitted with 139 total observations, 71 right-censored observations>

<lifelines.StatisticalResult: proportional_hazard_test>
    time_transform = rank
 null_distribution = chi squared
degrees_of_freedom = 1
             model = <lifelines.CoxPHFitter: fitted with 139 total observations, 71 right-censored observations>
         test_name = proportional_hazard_test

---
                                                       test_statistic    p  -log2(p)
LBP_003_CT                                                       0.01 0.91      0.13
LBP_003_PET                                                      0.00 0.98      0.03
LBP_012_CT                                                       0.00 0.95      0.08
LBP_012_PET                                                      0.00 0.97      0.04
LBP_021_CT                                                       0.00 0.97      0.05
LBP_021_PET                                                      0.01 0.92      0.12
LBP_030_CT                                                       0.00 0.99      0.02
LBP_030_PET       

In [7]:
# Access the summary table
summary_table = results.summary

# Filter columns based on p-values lower than 0.05
significant_columns = summary_table[summary_table['p'] < 0.05].index

# Display significant columns
print("Columns with p-values < 0.05:")
print(significant_columns)

Columns with p-values < 0.05:
Index([], dtype='object')


In [8]:
df = pd.DataFrame(clinical_train)

# Fit Cox proportional hazards model
cph = CoxPHFitter(penalizer=0.1)
cph.fit(df, duration_col='DFS', event_col='event_DFS')

# Perform the proportional hazards assumption test
results = proportional_hazard_test(cph, df)
print(results)

<lifelines.CoxPHFitter: fitted with 139 total observations, 71 right-censored observations>

<lifelines.StatisticalResult: proportional_hazard_test>
    time_transform = rank
 null_distribution = chi squared
degrees_of_freedom = 1
             model = <lifelines.CoxPHFitter: fitted with 139 total observations, 71 right-censored observations>
         test_name = proportional_hazard_test

---
                                                       test_statistic      p  -log2(p)
LBP_003_CT                                                       0.06   0.81      0.31
LBP_003_PET                                                      0.76   0.38      1.38
LBP_012_CT                                                       0.13   0.72      0.48
LBP_012_PET                                                      0.27   0.60      0.73
LBP_021_CT                                                       0.56   0.45      1.14
LBP_021_PET                                                      0.00   0.97      0.04
LBP_030_CT                                                       0.04   0.83      0.26
LB

In [9]:
# Access the summary table
summary_table = results.summary

# Filter columns based on p-values lower than 0.05
significant_columns = summary_table[summary_table['p'] < 0.05].index

# Display significant columns
print("Columns with p-values < 0.05:")
print(significant_columns)

Columns with p-values < 0.05:
Index(['charlson', 'glszm_LargeAreaEmphasis_CT_c16',
       'glszm_LargeAreaLowGrayLevelEmphasis_CT_c16',
       'glszm_ZoneVariance_CT_c16', 'ngtdm_Busyness_d_1_PET_b2'],
      dtype='object')


## Test dataset: MAASTRO 

In [10]:
(MAASTRO_D3['patient_id'] == MAASTRO_OS_target['patient_id']).sum()

99

In [11]:
# Rename the column name of response_MAASTRO 
response_MAASTRO.rename(columns = {'Index' : 'patient_id'}, inplace = True)

In [12]:
# Need to choose patient_id from MAASTRO_D3 in response_MAASTRO
data = list(MAASTRO_D3['patient_id'])
mask = response_MAASTRO['patient_id'].isin(data)
response_MAASTRO = response_MAASTRO[mask] 
response_MAASTRO

,patient_id,OS,OS_event,LRC,LRC_event,DFS,DFS_event
0,1,62.43,0.0,62.43,0.0,62.43,0.0
1,2,60.00,0.0,60.00,0.0,60.00,0.0
2,3,44.43,1.0,8.83,1.0,8.83,1.0
3,4,37.20,1.0,19.37,0.0,19.73,1.0
5,6,59.23,0.0,59.23,0.0,59.23,0.0
...,...,...,...,...,...,...,...
109,110,19.00,1.0,13.27,1.0,13.27,1.0
110,111,85.87,1.0,82.83,0.0,85.87,1.0
111,112,42.87,0.0,42.87,0.0,42.87,0.0
112,113,58.93,0.0,58.93,0.0,58.93,0.0


In [13]:
# Merge MAASTRO_D3 with response_MAASTRO
clinical_test = pd.merge(MAASTRO_D3, response_MAASTRO, on='patient_id', how='inner')
clinical_test = clinical_test.loc[:, ~clinical_test.columns.isin(['OS', 'OS_event', 'LRC', 'LRC_event'])]
clinical_test

,patient_id,age,female,cavum_oris,oropharynx,hypopharynx,larynx,histgrade_high,hpv_related,charlson,...,LBP_021_PET,LBP_030_PET,LBP_102_PET,LBP_111_PET,LBP_120_PET,LBP_201_PET,LBP_210_PET,LBP_300_PET,DFS,DFS_event
0,1,55,0,0,1,0,0,1,1,1,...,0.028808,0.837387,0.000026,0.001705,0.122209,0.000026,0.008291,0.000341,62.43,0.0
1,2,55,0,0,1,0,0,0,0,0,...,0.049615,0.806842,0.000167,0.002958,0.128976,0.000167,0.008148,0.000335,60.00,0.0
2,3,55,0,0,1,0,0,0,0,1,...,0.019514,0.830272,0.000057,0.001831,0.137282,0.000000,0.008641,0.000229,8.83,1.0
3,4,61,1,0,0,0,1,1,0,1,...,0.047155,0.760949,0.000000,0.003597,0.171595,0.000080,0.013187,0.000240,19.73,1.0
4,6,70,0,0,1,0,0,1,1,1,...,0.033990,0.824865,0.000000,0.002116,0.128134,0.000035,0.009379,0.000529,59.23,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
94,110,66,1,0,0,0,1,0,0,0,...,0.017489,0.834590,0.000000,0.001321,0.137194,0.000078,0.008706,0.000078,13.27,1.0
95,111,63,0,0,0,0,1,0,0,1,...,0.029979,0.821431,0.000000,0.000543,0.137620,0.000054,0.008907,0.000489,85.87,1.0
96,112,63,0,0,1,0,0,1,1,1,...,0.017354,0.859957,0.000000,0.000988,0.115099,0.000028,0.005700,0.000141,42.87,0.0
97,113,54,0,0,1,0,0,1,1,0,...,0.025399,0.847908,0.000000,0.001487,0.117654,0.000000,0.005759,0.000038,58.93,0.0


In [14]:
# Drop patient_id column
clinical_test = clinical_test.drop('patient_id', axis=1)

In [15]:
# Some rows have null values in OS, OS_event -> Remove those rows
clinical_test[clinical_test.isnull().any(axis=1)]
clinical_test = clinical_test.dropna(how='any',axis=0) 

,age,female,cavum_oris,oropharynx,hypopharynx,larynx,histgrade_high,hpv_related,charlson,pack_years,...,LBP_021_PET,LBP_030_PET,LBP_102_PET,LBP_111_PET,LBP_120_PET,LBP_201_PET,LBP_210_PET,LBP_300_PET,DFS,DFS_event


In [16]:
# Set X
X = clinical_train.loc[:, ~clinical_train.columns.isin(['DFS', 'event_DFS'])]

# Set y 
y = clinical_train.loc[:, ['DFS', 'event_DFS']]

In [17]:
# y into array 
lists = [] 
for i, j in zip(y['event_DFS'], y['DFS']): 
    lists.append((i, j))

y = np.array(lists, dtype=[('status', bool), ('time', np.int32)])

In [18]:
# Shape
print('X_train: ', X.shape)
print('y_train: ', y.shape)

X_train:  (139, 388)
y_train:  (139,)


In [19]:
# Change the name of a column 'DFS_event' in the clincial_test 
clinical_test.rename(columns = {'DFS_event' : 'event_DFS'}, inplace = True)

In [20]:
# Set X
X_MAASTRO = clinical_test.loc[:, ~clinical_test.columns.isin(['DFS', 'event_DFS'])]

# Set y_MAASTRO
y_MAASTRO = clinical_test.loc[:, ['DFS', 'event_DFS']]

# Change y_MAASTRO into array 
lists = [] 
for i, j in zip(y_MAASTRO['event_DFS'], y_MAASTRO['DFS']): 
    lists.append((i, j))

y_MAASTRO = np.array(lists, dtype=[('status', bool), ('time', np.int32)])

clinical_test.shape

(99, 390)

# Standardization

In [21]:
# Copy the original X for later 
original_X = X.copy()

In [22]:
# Standardize the data 
## Save the column and index 
X_columns = X.columns 
X_index = X.index

## Standardize the data but not the categorical columns
categorical_columns = ['female', 
                        'cavum_oris',
                        'oropharynx',
                        'hypopharynx',
                        'larynx',
                        'histgrade_high',
                        'hpv_related',
                        'charlson',
                        'uicc8_III-IV']
### Separate the categorical and non-categorical columns
X_categorical = X[categorical_columns]
X_numeric = X.drop(categorical_columns, axis=1)
X_numeric_columns = X_numeric.columns
X_numeric_index = X_numeric.index

### Standardize non-categorical and then concat with the categorical
scaler = StandardScaler()  
X_numeric_std = scaler.fit_transform(X_numeric)
X_numeric_std = pd.DataFrame(X_numeric_std, columns=X_numeric_columns, index=X_numeric_index)
X_std = pd.concat([X_categorical, X_numeric_std], axis=1)

## Sort the order of the columns as it was in the clinical train 
X_std = X_std[original_X.columns]

In [23]:
# Divide the X_MAASTRO into numerical part and categorical part 
X_MAASTRO_categorical = X_MAASTRO[categorical_columns]
X_MAASTRO_numeric = X_MAASTRO.drop(categorical_columns, axis=1)

# Save the column name and index of the numeric part
X_MAASTRO_numeric_columns = X_MAASTRO_numeric.columns
X_MAASTRO_numeric_index = X_MAASTRO_numeric.index

In [24]:
# Standardize the numeric part 
X_MAASTRO_numeric_std = scaler.transform(X_MAASTRO_numeric)

# Change the standardized part into a dataframe 
X_MAASTRO_numeric_std = pd.DataFrame(X_MAASTRO_numeric_std, columns=X_MAASTRO_numeric_columns, index=X_MAASTRO_numeric_index)

# Concat the standardized part with the categorical part 
X_MAASTRO_std = pd.concat([X_MAASTRO_categorical, X_MAASTRO_numeric_std], axis=1)

# Change the column order of X_MAASTRO_std
X_MAASTRO_std = X_MAASTRO_std[original_X.columns]

In [25]:
# Saving the data 
X_new = X 
X_new_std = X_std 
MAASTRO_new = X_MAASTRO 
MAASTRO_new_std = X_MAASTRO_std 

# Modelling 

### 1. CoxPHSurvivalAnalysis

#### Train

In [26]:
# Setting the y format for skf below  
y = clinical_train[['DFS', 'event_DFS']]

# Running to optuna for hyperparameter tuning 
def create_objective(model_class, metric, X, y):
    def objective(trial): 
        # Create and fit survival model 
        model = model_class()
        
        scores = [] 

        skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=123)
        
        for k, (train_index, test_index) in enumerate(skf.split(X, y.iloc[:, 1])): 
            X_train, X_test = X.iloc[train_index], X.iloc[test_index]
            y_train_df, y_test_df = y.iloc[train_index], y.iloc[test_index]
            
            # y_train into array 
            y_train = [] 
            for i, j in zip(y_train_df['event_DFS'], y_train_df['DFS']): 
                y_train.append((i, j))
            y_train = np.array(y_train, dtype=[('status', bool), ('time', np.int32)])

            # y_test into array
            y_test = [] 
            for i, j in zip(y_test_df['event_DFS'], y_test_df['DFS']): 
                y_test.append((i, j))
            y_test = np.array(y_test, dtype=[('status', bool), ('time', np.int32)])

            # Standardization
            excluded_columns = ['female', 
                                'cavum_oris',
                                'oropharynx',
                                'hypopharynx',
                                'larynx',
                                'histgrade_high',
                                'hpv_related',
                                'charlson',
                                'uicc8_III-IV'
                               ]
            excluded_columns = set(excluded_columns).intersection(X.columns)

            scaler = StandardScaler() 
            X_train_included = X_train.drop(excluded_columns, axis=1)
            X_test_included = X_test.drop(excluded_columns, axis=1)
                        
            if not X_train_included.empty and not X_test_included.empty:
                X_train_included_std = scaler.fit_transform(X_train_included)
                X_test_included_std = scaler.transform(X_test_included)
                
                # Concatenation
                X_train_std_df = pd.DataFrame(X_train_included_std, columns=X_train_included.columns, index=X_train_included.index)
                X_train_std = pd.concat([X_train_std_df, X_train[excluded_columns]], axis=1)

                X_test_std_df = pd.DataFrame(X_test_included_std, columns=X_test_included.columns, index=X_test_included.index)
                X_test_std = pd.concat([X_test_std_df, X_test[excluded_columns]], axis=1)
            
            else: 
                X_train_std = X_train
                X_test_std = X_test 
            
            model.fit(X_train_std, y_train)

            if metric == "c-index":
                # Make predictions using C-index 
                c_index_score = model.score(X_test_std, y_test)
                scores.append(c_index_score)
                print(f"Fold {k + 1} C-index: {c_index_score}")
                
            elif metric == "ibs":
                # Make predictions using IBS 
                lower, upper = np.percentile(y_test["time"], [10, 90])    
                times = np.arange(lower, upper)
                surv_prob = np.row_stack([fn(times) for fn in model.predict_survival_function(X_test_std)])
                ibs = integrated_brier_score(y_test, y_test, surv_prob, times)
                scores.append(ibs)
                print(f"Fold {k + 1} IBS: {ibs}")
            else:
                raise ValueError("Invalid metric. Use 'C-index' or 'ibs'.")
        
        # Return the mean of scores
        return np.mean(scores)
    
    return objective

# C-index
study_cindex = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=123))
objective_cindex = create_objective(CoxPHSurvivalAnalysis, "c-index", X_new, y)
study_cindex.optimize(objective_cindex, n_trials=1, show_progress_bar=True)
print("\n")
print("* Best trial for C-index: \n", study_cindex.best_trial)
print("\n")
print("* Best Score for C-index: \n", study_cindex.best_value)

# Example usage for IBS
study_ibs = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=123))
objective_ibs = create_objective(CoxPHSurvivalAnalysis, "ibs", X_new, y)
study_ibs.optimize(objective_ibs, n_trials=1, show_progress_bar=True)
print("\n")
print("* Best trial for IBS: \n", study_ibs.best_trial)
print("\n")
print("* Best Score for IBS: \n", study_ibs.best_value)

[I 2024-04-16 18:35:39,653] A new study created in memory with name: no-name-30ce43f1-6d8c-4046-a577-c0cac58a26d8
python(25199) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.


  0%|          | 0/1 [00:00<?, ?it/s]

[W 2024-04-16 18:35:41,472] Trial 0 failed with parameters: {} because of the following error: LinAlgError('Matrix is singular.').
Traceback (most recent call last):
  File "/Users/minjeongcheon/opt/anaconda3/lib/python3.9/site-packages/optuna/study/_optimize.py", line 200, in _run_trial
    value_or_values = func(trial)
  File "/var/folders/64/jkqp6xyx2hj50dmd2pqfm3780000gn/T/ipykernel_24196/2750526300.py", line 62, in objective
    model.fit(X_train_std, y_train)
  File "/Users/minjeongcheon/opt/anaconda3/lib/python3.9/site-packages/sksurv/linear_model/coxph.py", line 449, in fit
    delta = solve(
  File "/Users/minjeongcheon/opt/anaconda3/lib/python3.9/site-packages/scipy/linalg/_basic.py", line 220, in solve
    _solve_check(n, info)
  File "/Users/minjeongcheon/opt/anaconda3/lib/python3.9/site-packages/scipy/linalg/_basic.py", line 29, in _solve_check
    raise LinAlgError('Matrix is singular.')
numpy.linalg.LinAlgError: Matrix is singular.
[W 2024-04-16 18:35:41,492] Trial 0 fai

LinAlgError: Matrix is singular.

In [27]:
# Setting a dictionary to save the train results 
train_cindex = {} 
train_ibs = {} 

In [28]:
# Saving the values to the dictionary 
train_cindex['CoxPH'] = np.round(study_cindex.best_value, 3)
train_ibs['CoxPH'] = np.round(study_ibs.best_value, 3)

ValueError: No trials are completed yet.

In [29]:
print("train_cindex: ", np.round(study_cindex.best_value, 3))
print("train_ibs: ", np.round(study_ibs.best_value, 3))

ValueError: No trials are completed yet.

#### Test

In [30]:
# y into array 
lists = [] 
for i, j in zip(y['event_DFS'], y['DFS']): 
    lists.append((i, j))

y = np.array(lists, dtype=[('status', bool), ('time', np.int32)])

In [31]:
# Test on MAASTRO 
cph = CoxPHSurvivalAnalysis()

cph.fit(X_new_std, y)

# Save C-index 
c_index = cph.score(MAASTRO_new_std, y_MAASTRO)
c_index = np.round(c_index, 3)
print('Concordance index:', c_index)

# Save IBS 
lower, upper = np.percentile(y_MAASTRO["time"], [10, 90])
times = np.arange(lower, upper)
surv_prob = np.row_stack([fn(times) for fn in cph.predict_survival_function(MAASTRO_new_std)])
ibs = integrated_brier_score(y_MAASTRO, y_MAASTRO, surv_prob, times)
ibs = np.round(ibs, 3)
print('IBS score:', ibs)

LinAlgError: Matrix is singular.

In [35]:
# Setting a dictionary to save the test results 
test_cindex = {} 
test_ibs = {} 

In [32]:
# Saving the values to the dictionary 
test_cindex['CoxPH'] = c_index
test_ibs['CoxPH'] = ibs

NameError: name 'c_index' is not defined

### 2. CoxnetSurvivalAnalysis - Ridge

#### Train

In [36]:
# Setting the y format 
y = clinical_train[['DFS', 'event_DFS']]

# Running to optuna for hyperparameter tuning 
def create_objective(model_class, metric, X, y):
    def objective(trial): 
        # Create and fit survival model 
        model = model_class(l1_ratio=0.0000001, 
                            fit_baseline_model=True)
        
        scores = [] 

        skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=123)
        
        for k, (train_index, test_index) in enumerate(skf.split(X, y.iloc[:, 1])): 
            X_train, X_test = X.iloc[train_index], X.iloc[test_index]
            y_train_df, y_test_df = y.iloc[train_index], y.iloc[test_index]
            
            # y_train into array 
            y_train = [] 
            for i, j in zip(y_train_df['event_DFS'], y_train_df['DFS']): 
                y_train.append((i, j))
            y_train = np.array(y_train, dtype=[('status', bool), ('time', np.int32)])

            # y_test into array
            y_test = [] 
            for i, j in zip(y_test_df['event_DFS'], y_test_df['DFS']): 
                y_test.append((i, j))
            y_test = np.array(y_test, dtype=[('status', bool), ('time', np.int32)])

            # Standardization
            excluded_columns = ['female', 
                                'cavum_oris',
                                'oropharynx',
                                'hypopharynx',
                                'larynx',
                                'histgrade_high',
                                'hpv_related',
                                'charlson',
                                'uicc8_III-IV'
                               ]
            excluded_columns = set(excluded_columns).intersection(X.columns)

            scaler = StandardScaler() 
            X_train_included = X_train.drop(excluded_columns, axis=1)
            X_test_included = X_test.drop(excluded_columns, axis=1)
                        
            if not X_train_included.empty and not X_test_included.empty:
                X_train_included_std = scaler.fit_transform(X_train_included)
                X_test_included_std = scaler.transform(X_test_included)
                
                # Concatenation
                X_train_std_df = pd.DataFrame(X_train_included_std, columns=X_train_included.columns, index=X_train_included.index)
                X_train_std = pd.concat([X_train_std_df, X_train[excluded_columns]], axis=1)

                X_test_std_df = pd.DataFrame(X_test_included_std, columns=X_test_included.columns, index=X_test_included.index)
                X_test_std = pd.concat([X_test_std_df, X_test[excluded_columns]], axis=1)
            
            else: 
                X_train_std = X_train
                X_test_std = X_test 
            
            model.fit(X_train_std, y_train)

            if metric == "c-index":
                # Make predictions using C-index 
                c_index_score = model.score(X_test_std, y_test)
                scores.append(c_index_score)
                print(f"Fold {k + 1} C-index: {c_index_score}")
                
            elif metric == "ibs":
                # Make predictions using IBS 
                lower, upper = np.percentile(y_test["time"], [10, 90])    
                times = np.arange(lower, upper)
                surv_prob = np.row_stack([fn(times) for fn in model.predict_survival_function(X_test_std)])
                ibs = integrated_brier_score(y_test, y_test, surv_prob, times)
                scores.append(ibs)
                print(f"Fold {k + 1} IBS: {ibs}")
            else:
                raise ValueError("Invalid metric. Use 'C-index' or 'ibs'.")
        
        # Return the mean of scores
        return np.mean(scores)
    
    return objective

# C-index
study_cindex = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=123))
objective_cindex = create_objective(CoxnetSurvivalAnalysis, "c-index", X_new, y)
study_cindex.optimize(objective_cindex, n_trials=1, show_progress_bar=True)
print("\n")
print("* Best trial for C-index: \n", study_cindex.best_trial)
print("\n")
print("* Best Score for C-index: \n", study_cindex.best_value)

# Example usage for IBS
study_ibs = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=123))
objective_ibs = create_objective(CoxnetSurvivalAnalysis, "ibs", X_new, y)
study_ibs.optimize(objective_ibs, n_trials=1, show_progress_bar=True)
print("\n")
print("* Best trial for IBS: \n", study_ibs.best_trial)
print("\n")
print("* Best Score for IBS: \n", study_ibs.best_value)


[I 2024-04-16 20:21:22,369] A new study created in memory with name: no-name-df37f121-e1b2-4210-b5cb-556856467b48


  0%|          | 0/1 [00:00<?, ?it/s]

[I 2024-04-16 20:21:22,644] A new study created in memory with name: no-name-4305f0e0-6ce8-4ee1-8ba8-6aa6cc48aefa


Fold 1 C-index: 0.5776892430278885
Fold 2 C-index: 0.7267441860465116
Fold 3 C-index: 0.5680851063829787
Fold 4 C-index: 0.6615969581749049
Fold 5 C-index: 0.630901287553648
[I 2024-04-16 20:21:22,639] Trial 0 finished with value: 0.6330033562371864 and parameters: {}. Best is trial 0 with value: 0.6330033562371864.


* Best trial for C-index: 
 FrozenTrial(number=0, state=TrialState.COMPLETE, values=[0.6330033562371864], datetime_start=datetime.datetime(2024, 4, 16, 20, 21, 22, 431844), datetime_complete=datetime.datetime(2024, 4, 16, 20, 21, 22, 639209), params={}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={}, trial_id=0, value=None)


* Best Score for C-index: 
 0.6330033562371864


  0%|          | 0/1 [00:00<?, ?it/s]

Fold 1 IBS: 0.24724709766223185
Fold 2 IBS: 0.2320398804259208
Fold 3 IBS: 0.22898186781735627
Fold 4 IBS: 0.24197476635329754
Fold 5 IBS: 0.22939558893495351
[I 2024-04-16 20:21:22,902] Trial 0 finished with value: 0.235927840238752 and parameters: {}. Best is trial 0 with value: 0.235927840238752.


* Best trial for IBS: 
 FrozenTrial(number=0, state=TrialState.COMPLETE, values=[0.235927840238752], datetime_start=datetime.datetime(2024, 4, 16, 20, 21, 22, 679406), datetime_complete=datetime.datetime(2024, 4, 16, 20, 21, 22, 902279), params={}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={}, trial_id=0, value=None)


* Best Score for IBS: 
 0.235927840238752


In [37]:
train_cindex['CoxRidge'] = np.round(study_cindex.best_value, 3)
train_ibs['CoxRidge'] = np.round(study_ibs.best_value, 3)

In [38]:
print("train_cindex: ", np.round(study_cindex.best_value, 3))
print("train_ibs: ", np.round(study_ibs.best_value, 3))

train_cindex:  0.633
train_ibs:  0.236


#### Test

In [39]:
# y into array 
lists = [] 
for i, j in zip(y['event_DFS'], y['DFS']): 
    lists.append((i, j))

y = np.array(lists, dtype=[('status', bool), ('time', np.int32)])

In [40]:
# A function for building the best model with the best parameters 
def create_best_model(model_class, best_params):
    best_params['l1_ratio'] = 0.0000001
    best_params['fit_baseline_model']=True
    return model_class(**best_params)

# Set the best model 
best_model_cindex = create_best_model(CoxnetSurvivalAnalysis, 
                                      study_cindex.best_params)

# Train the best model for C-index on the whole dataset
best_model_cindex.fit(X_new_std, y)

# Evaluate the best model for C-index on MAASTRO dataset
c_index = best_model_cindex.score(MAASTRO_new_std, y_MAASTRO)
c_index = np.round(c_index, 3)
print("test_cindex :", c_index)

# Set the best model 
best_model_ibs = create_best_model(CoxnetSurvivalAnalysis, study_ibs.best_params)

# Train the best model for IBS on the whole dataset
best_model_ibs.fit(X_new_std, y)

# Evaluate the best model for IBS on MAASTRO dataset
lower, upper = np.percentile(y_MAASTRO["time"], [10, 90])
times = np.arange(lower, upper)
surv_prob = np.row_stack([fn(times) for fn in best_model_ibs.predict_survival_function(MAASTRO_new_std)])
ibs = integrated_brier_score(y_MAASTRO, y_MAASTRO, surv_prob, times)
ibs = np.round(ibs, 3)
print("test_ibs: ", ibs)

CoxnetSurvivalAnalysis(fit_baseline_model=True, l1_ratio=1e-07)

test_cindex : 0.542


CoxnetSurvivalAnalysis(fit_baseline_model=True, l1_ratio=1e-07)

test_ibs:  0.229


In [41]:
# Saving the values to the dictionary 
test_cindex['CoxRidge'] = c_index
test_ibs['CoxRidge'] = ibs

### 3. CoxnetSurvivalAnalysis - Lasso

#### Train

In [42]:
# Setting the y format 
y = clinical_train[['DFS', 'event_DFS']]

def create_objective(model_class, metric, X, y):
    def objective(trial): 
        # Create and fit survival model 
        model = model_class(l1_ratio=1, 
                            fit_baseline_model=True)
        
        scores = [] 

        skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=123)
        
        for k, (train_index, test_index) in enumerate(skf.split(X, y.iloc[:, 1])): 
            X_train, X_test = X.iloc[train_index], X.iloc[test_index]
            y_train_df, y_test_df = y.iloc[train_index], y.iloc[test_index]
            
            # y_train into array 
            y_train = [] 
            for i, j in zip(y_train_df['event_DFS'], y_train_df['DFS']): 
                y_train.append((i, j))
            y_train = np.array(y_train, dtype=[('status', bool), ('time', np.int32)])

            # y_test into array
            y_test = [] 
            for i, j in zip(y_test_df['event_DFS'], y_test_df['DFS']): 
                y_test.append((i, j))
            y_test = np.array(y_test, dtype=[('status', bool), ('time', np.int32)])

            # Standardization
            excluded_columns = ['female', 
                                'cavum_oris',
                                'oropharynx',
                                'hypopharynx',
                                'larynx',
                                'histgrade_high',
                                'hpv_related',
                                'charlson',
                                'uicc8_III-IV'
                               ]
            excluded_columns = set(excluded_columns).intersection(X.columns)

            scaler = StandardScaler() 
            X_train_included = X_train.drop(excluded_columns, axis=1)
            X_test_included = X_test.drop(excluded_columns, axis=1)
                        
            if not X_train_included.empty and not X_test_included.empty:
                X_train_included_std = scaler.fit_transform(X_train_included)
                X_test_included_std = scaler.transform(X_test_included)
                
                # Concatenation
                X_train_std_df = pd.DataFrame(X_train_included_std, columns=X_train_included.columns, index=X_train_included.index)
                X_train_std = pd.concat([X_train_std_df, X_train[excluded_columns]], axis=1)

                X_test_std_df = pd.DataFrame(X_test_included_std, columns=X_test_included.columns, index=X_test_included.index)
                X_test_std = pd.concat([X_test_std_df, X_test[excluded_columns]], axis=1)
            
            else: 
                X_train_std = X_train
                X_test_std = X_test 
            
            model.fit(X_train_std, y_train)

            if metric == "c-index":
                # Make predictions using C-index 
                c_index_score = model.score(X_test_std, y_test)
                scores.append(c_index_score)
                print(f"Fold {k + 1} C-index: {c_index_score}")
                
            elif metric == "ibs":
                # Make predictions using IBS 
                lower, upper = np.percentile(y_test["time"], [10, 90])    
                times = np.arange(lower, upper)
                cox_surv_prob = np.row_stack([fn(times) for fn in model.predict_survival_function(X_test_std)])
                ibs = integrated_brier_score(y_test, y_test, cox_surv_prob, times)
                scores.append(ibs)
                print(f"Fold {k + 1} IBS: {ibs}")
            else:
                raise ValueError("Invalid metric. Use 'C-index' or 'ibs'.")
        
        # Return the mean of scores
        return np.mean(scores)
    
    return objective

# C-index
study_cindex = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=123))
objective_cindex = create_objective(CoxnetSurvivalAnalysis, "c-index", X_new, y)
study_cindex.optimize(objective_cindex, n_trials=1, show_progress_bar=True)
print("\n")
print("* Best trial for C-index: \n", study_cindex.best_trial)
print("\n")
print("* Best Score for C-index: \n", study_cindex.best_value)

# Example usage for IBS
study_ibs = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=123))
objective_ibs = create_objective(CoxnetSurvivalAnalysis, "ibs", X_new, y)
study_ibs.optimize(objective_ibs, n_trials=1, show_progress_bar=True)
print("\n")
print("* Best trial for IBS: \n", study_ibs.best_trial)
print("\n")
print("* Best Score for IBS: \n", study_ibs.best_value)


[I 2024-04-16 20:21:24,727] A new study created in memory with name: no-name-b87c1f42-3af2-4124-999d-3326178a69d3


  0%|          | 0/1 [00:00<?, ?it/s]

Fold 1 C-index: 0.43824701195219123
Fold 2 C-index: 0.6317829457364341
Fold 3 C-index: 0.4553191489361702
Fold 4 C-index: 0.4448669201520912


[I 2024-04-16 20:21:26,576] A new study created in memory with name: no-name-2bd4f78b-309c-4ca5-b53b-13f1aff7614f


Fold 5 C-index: 0.5622317596566524
[I 2024-04-16 20:21:26,562] Trial 0 finished with value: 0.5064895572867079 and parameters: {}. Best is trial 0 with value: 0.5064895572867079.


* Best trial for C-index: 
 FrozenTrial(number=0, state=TrialState.COMPLETE, values=[0.5064895572867079], datetime_start=datetime.datetime(2024, 4, 16, 20, 21, 24, 754268), datetime_complete=datetime.datetime(2024, 4, 16, 20, 21, 26, 562524), params={}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={}, trial_id=0, value=None)


* Best Score for C-index: 
 0.5064895572867079


  0%|          | 0/1 [00:00<?, ?it/s]

Fold 1 IBS: 0.47722249153378843
Fold 2 IBS: 0.3559505337162963
Fold 3 IBS: 0.47560772148855335
Fold 4 IBS: 0.4696450550522016
Fold 5 IBS: 0.3821180064513121
[I 2024-04-16 20:21:28,369] Trial 0 finished with value: 0.4321087616484304 and parameters: {}. Best is trial 0 with value: 0.4321087616484304.


* Best trial for IBS: 
 FrozenTrial(number=0, state=TrialState.COMPLETE, values=[0.4321087616484304], datetime_start=datetime.datetime(2024, 4, 16, 20, 21, 26, 616131), datetime_complete=datetime.datetime(2024, 4, 16, 20, 21, 28, 369773), params={}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={}, trial_id=0, value=None)


* Best Score for IBS: 
 0.4321087616484304


In [43]:
train_cindex['CoxLasso'] = np.round(study_cindex.best_value, 3)
train_ibs['CoxLasso'] = np.round(study_ibs.best_value, 3)

In [44]:
print("train_cindex: ", np.round(study_cindex.best_value, 3))
print("train_ibs: ", np.round(study_ibs.best_value, 3))

train_cindex:  0.506
train_ibs:  0.432


#### Test

In [45]:
# y into array 
lists = [] 
for i, j in zip(y['event_DFS'], y['DFS']): 
    lists.append((i, j))

y = np.array(lists, dtype=[('status', bool), ('time', np.int32)])

In [46]:
# A function for building the best model with the best parameters 
def create_best_model(model_class, best_params):
    best_params['l1_ratio'] = 1
    best_params['fit_baseline_model']=True
    return model_class(**best_params)

# Set the best model 
best_model_cindex = create_best_model(CoxnetSurvivalAnalysis, 
                                      study_cindex.best_params)

# Train the best model for C-index on the whole dataset
best_model_cindex.fit(X_new_std, y)

# Evaluate the best model for C-index on MAASTRO dataset
c_index = best_model_cindex.score(MAASTRO_new_std, y_MAASTRO)
c_index = np.round(c_index, 3)
print("test_cindex :", c_index)

# Set the best model 
best_model_ibs = create_best_model(CoxnetSurvivalAnalysis, study_ibs.best_params)

# Train the best model for IBS on the whole dataset
best_model_ibs.fit(X_new_std, y)

# Evaluate the best model for IBS on MAASTRO dataset
lower, upper = np.percentile(y_MAASTRO["time"], [10, 90])
times = np.arange(lower, upper)
surv_prob = np.row_stack([fn(times) for fn in best_model_ibs.predict_survival_function(MAASTRO_new_std)])
ibs = integrated_brier_score(y_MAASTRO, y_MAASTRO, surv_prob, times)
ibs = np.round(ibs, 3)
print("test_ibs: ", ibs)

CoxnetSurvivalAnalysis(fit_baseline_model=True, l1_ratio=1)

test_cindex : 0.545


CoxnetSurvivalAnalysis(fit_baseline_model=True, l1_ratio=1)

test_ibs:  0.431


In [47]:
# Saving the values to the dictionary 
test_cindex['CoxLasso'] = c_index
test_ibs['CoxLasso'] = ibs

### 4. CoxnetSurvivalAnalysis - ElasticNet

#### Train

In [48]:
# Setting the y format 
y = clinical_train[['DFS', 'event_DFS']]

def create_objective(model_class, metric, X, y):
    def objective(trial): 
        # Suggest values for hyperparameters
        l1_ratio = trial.suggest_float("l1_ratio", 0.0001, 1)
        
        # Create and fit survival model 
        model = model_class(l1_ratio=l1_ratio, 
                           fit_baseline_model=True)
        
        scores = [] 

        skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=123)
        
        for k, (train_index, test_index) in enumerate(skf.split(X, y.iloc[:, 1])): 
            X_train, X_test = X.iloc[train_index], X.iloc[test_index]
            y_train_df, y_test_df = y.iloc[train_index], y.iloc[test_index]
            
            # y_train into array 
            y_train = [] 
            for i, j in zip(y_train_df['event_DFS'], y_train_df['DFS']): 
                y_train.append((i, j))
            y_train = np.array(y_train, dtype=[('status', bool), ('time', np.int32)])

            # y_test into array
            y_test = [] 
            for i, j in zip(y_test_df['event_DFS'], y_test_df['DFS']): 
                y_test.append((i, j))
            y_test = np.array(y_test, dtype=[('status', bool), ('time', np.int32)])

            # Standardization
            excluded_columns = ['female', 
                                'cavum_oris',
                                'oropharynx',
                                'hypopharynx',
                                'larynx',
                                'histgrade_high',
                                'hpv_related',
                                'charlson',
                                'uicc8_III-IV'
                               ]
            excluded_columns = set(excluded_columns).intersection(X.columns)

            scaler = StandardScaler() 
            X_train_included = X_train.drop(excluded_columns, axis=1)
            X_test_included = X_test.drop(excluded_columns, axis=1)
                        
            if not X_train_included.empty and not X_test_included.empty:
                X_train_included_std = scaler.fit_transform(X_train_included)
                X_test_included_std = scaler.transform(X_test_included)
                
                # Concatenation
                X_train_std_df = pd.DataFrame(X_train_included_std, columns=X_train_included.columns, index=X_train_included.index)
                X_train_std = pd.concat([X_train_std_df, X_train[excluded_columns]], axis=1)

                X_test_std_df = pd.DataFrame(X_test_included_std, columns=X_test_included.columns, index=X_test_included.index)
                X_test_std = pd.concat([X_test_std_df, X_test[excluded_columns]], axis=1)
            
            else: 
                X_train_std = X_train
                X_test_std = X_test 
            
            model.fit(X_train_std, y_train)

            if metric == "c-index":
                # Make predictions using C-index 
                c_index_score = model.score(X_test_std, y_test)
                scores.append(c_index_score)
                print(f"Fold {k + 1} C-index: {c_index_score}")
                
            elif metric == "ibs":
                # Make predictions using IBS 
                lower, upper = np.percentile(y_test["time"], [10, 90])    
                times = np.arange(lower, upper)
                cox_surv_prob = np.row_stack([fn(times) for fn in model.predict_survival_function(X_test_std)])
                ibs = integrated_brier_score(y_test, y_test, cox_surv_prob, times)
                scores.append(ibs)
                print(f"Fold {k + 1} IBS: {ibs}")
            else:
                raise ValueError("Invalid metric. Use 'C-index' or 'ibs'.")
        
        # Return the mean of scores
        return np.mean(scores)
    
    return objective

# C-index
study_cindex = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=123))
objective_cindex = create_objective(CoxnetSurvivalAnalysis, "c-index", X_new, y)
study_cindex.optimize(objective_cindex, n_trials=100, show_progress_bar=True)
print("\n")
print("* Best trial for C-index: \n", study_cindex.best_trial)
print("\n")
print("* Best Score for C-index: \n", study_cindex.best_value)

# Example usage for IBS
study_ibs = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=123))
objective_ibs = create_objective(CoxnetSurvivalAnalysis, "ibs", X_new, y)
study_ibs.optimize(objective_ibs, n_trials=100, show_progress_bar=True)
print("\n")
print("* Best trial for IBS: \n", study_ibs.best_trial)
print("\n")
print("* Best Score for IBS: \n", study_ibs.best_value)


[I 2024-04-16 20:21:29,301] A new study created in memory with name: no-name-68c40b1f-e82e-4f9e-814b-d68f3f79bd8c


  0%|          | 0/100 [00:00<?, ?it/s]

Fold 1 C-index: 0.41434262948207173
Fold 2 C-index: 0.5891472868217055
Fold 3 C-index: 0.425531914893617
Fold 4 C-index: 0.5019011406844106
Fold 5 C-index: 0.5493562231759657
[I 2024-04-16 20:21:30,769] Trial 0 finished with value: 0.49605583901155403 and parameters: {'l1_ratio': 0.6964995386793018}. Best is trial 0 with value: 0.49605583901155403.
Fold 1 C-index: 0.43824701195219123
Fold 2 C-index: 0.5852713178294574
Fold 3 C-index: 0.44680851063829785
Fold 4 C-index: 0.5285171102661597
Fold 5 C-index: 0.5407725321888412
[I 2024-04-16 20:21:32,046] Trial 1 finished with value: 0.5079232965749896 and parameters: {'l1_ratio': 0.28621072101688444}. Best is trial 1 with value: 0.5079232965749896.
Fold 1 C-index: 0.43824701195219123
Fold 2 C-index: 0.5852713178294574
Fold 3 C-index: 0.44680851063829785
Fold 4 C-index: 0.5361216730038023
Fold 5 C-index: 0.5407725321888412
[I 2024-04-16 20:21:33,304] Trial 2 finished with value: 0.509444209122518 and parameters: {'l1_ratio': 0.22692876841884

Fold 5 C-index: 0.630901287553648
[I 2024-04-16 20:21:54,641] Trial 23 finished with value: 0.6345047956387423 and parameters: {'l1_ratio': 0.002373744016739676}. Best is trial 18 with value: 0.6346095898895594.
Fold 1 C-index: 0.4342629482071713
Fold 2 C-index: 0.7170542635658915
Fold 3 C-index: 0.5617021276595745
Fold 4 C-index: 0.5513307984790875
Fold 5 C-index: 0.5493562231759657
[I 2024-04-16 20:21:55,438] Trial 24 finished with value: 0.562741272217538 and parameters: {'l1_ratio': 0.17890880660260244}. Best is trial 18 with value: 0.6346095898895594.
Fold 1 C-index: 0.44223107569721115
Fold 2 C-index: 0.5891472868217055
Fold 3 C-index: 0.42127659574468085
Fold 4 C-index: 0.5133079847908745
Fold 5 C-index: 0.5407725321888412
[I 2024-04-16 20:21:56,818] Trial 25 finished with value: 0.5013470950486626 and parameters: {'l1_ratio': 0.3507057512687868}. Best is trial 18 with value: 0.6346095898895594.
Fold 1 C-index: 0.4581673306772908
Fold 2 C-index: 0.7209302325581395
Fold 3 C-index

Fold 1 C-index: 0.4342629482071713
Fold 2 C-index: 0.7170542635658915
Fold 3 C-index: 0.5617021276595745
Fold 4 C-index: 0.6577946768060836
Fold 5 C-index: 0.5579399141630901
[I 2024-04-16 20:22:07,654] Trial 47 finished with value: 0.5857507860803622 and parameters: {'l1_ratio': 0.1298882306515754}. Best is trial 36 with value: 0.6380067098091695.
Fold 1 C-index: 0.4262948207171315
Fold 2 C-index: 0.5891472868217055
Fold 3 C-index: 0.42127659574468085
Fold 4 C-index: 0.49809885931558934
Fold 5 C-index: 0.5493562231759657
[I 2024-04-16 20:22:08,957] Trial 48 finished with value: 0.4968347571550146 and parameters: {'l1_ratio': 0.758842798046546}. Best is trial 36 with value: 0.6380067098091695.
Fold 1 C-index: 0.42231075697211157
Fold 2 C-index: 0.5813953488372093
Fold 3 C-index: 0.42127659574468085
Fold 4 C-index: 0.49809885931558934
Fold 5 C-index: 0.51931330472103
[I 2024-04-16 20:22:09,970] Trial 49 finished with value: 0.4884789731181242 and parameters: {'l1_ratio': 0.4477016529538

...

  0%|          | 0/100 [00:00<?, ?it/s]

Fold 1 IBS: 0.46400927837705297
Fold 2 IBS: 0.36326713437890934
Fold 3 IBS: 0.4495531678028146
Fold 4 IBS: 0.4230153149702206
Fold 5 IBS: 0.37804967345141277
[I 2024-04-16 20:22:33,780] Trial 0 finished with value: 0.41557891379608203 and parameters: {'l1_ratio': 0.6964995386793018}. Best is trial 0 with value: 0.41557891379608203.
Fold 1 IBS: 0.47153796586527336
Fold 2 IBS: 0.35072914215914724
Fold 3 IBS: 0.40203663354608876
Fold 4 IBS: 0.3933450035709491
Fold 5 IBS: 0.3584191801705698
[I 2024-04-16 20:22:35,351] Trial 1 finished with value: 0.39521358506240567 and parameters: {'l1_ratio': 0.28621072101688444}. Best is trial 1 with value: 0.39521358506240567.
Fold 1 IBS: 0.4683009668637043
Fold 2 IBS: 0.34248170696972846
Fold 3 IBS: 0.38200623914650983
Fold 4 IBS: 0.37349309893955285
Fold 5 IBS: 0.3516793263104152
[I 2024-04-16 20:22:36,473] Trial 2 finished with value: 0.3835922676459821 and parameters: {'l1_ratio': 0.22692876841884668}. Best is trial 2 with value: 0.3835922676459821

Fold 1 IBS: 0.44878590243432254
Fold 2 IBS: 0.22895499755289705
Fold 3 IBS: 0.2288138071100771
Fold 4 IBS: 0.23899859714513985
Fold 5 IBS: 0.2265399329750911
[I 2024-04-16 20:23:23,011] Trial 25 finished with value: 0.2744186474435055 and parameters: {'l1_ratio': 0.08890608406899088}. Best is trial 12 with value: 0.2351594079587081.
Fold 1 IBS: 0.4689238457546616
Fold 2 IBS: 0.3438648658019378
Fold 3 IBS: 0.3849804469982634
Fold 4 IBS: 0.3762695755182494
Fold 5 IBS: 0.3525388212473729
[I 2024-04-16 20:23:25,079] Trial 26 finished with value: 0.385315511064097 and parameters: {'l1_ratio': 0.23521758130887002}. Best is trial 12 with value: 0.2351594079587081.
Fold 1 IBS: 0.46457863126453197
Fold 2 IBS: 0.36642881617790757
Fold 3 IBS: 0.45258628016037145
Fold 4 IBS: 0.4335524959912747
Fold 5 IBS: 0.3890169998588207
[I 2024-04-16 20:23:28,162] Trial 27 finished with value: 0.4212326446905813 and parameters: {'l1_ratio': 0.8345729974660353}. Best is trial 12 with value: 0.2351594079587081.


Fold 1 IBS: 0.4528680686176761
Fold 2 IBS: 0.22855176102945338
Fold 3 IBS: 0.2287962292644494
Fold 4 IBS: 0.23870799813892157
Fold 5 IBS: 0.3210081231764322
[I 2024-04-16 20:23:45,959] Trial 50 finished with value: 0.2939864360453865 and parameters: {'l1_ratio': 0.10304820804750442}. Best is trial 49 with value: 0.2340790786483337.
Fold 1 IBS: 0.24587444933585
Fold 2 IBS: 0.2297735686866457
Fold 3 IBS: 0.22885322693415577
Fold 4 IBS: 0.23964835057429018
Fold 5 IBS: 0.2272685279911469
[I 2024-04-16 20:23:46,213] Trial 51 finished with value: 0.2342836247044177 and parameters: {'l1_ratio': 0.0622595569832182}. Best is trial 49 with value: 0.2340790786483337.
Fold 1 IBS: 0.46156717425811067
Fold 2 IBS: 0.2272903421113041
Fold 3 IBS: 0.22875104746177286
Fold 4 IBS: 0.34557212563307005
Fold 5 IBS: 0.3370205701521933
[I 2024-04-16 20:23:47,286] Trial 52 finished with value: 0.3200402519232902 and parameters: {'l1_ratio': 0.15249375205395843}. Best is trial 49 with value: 0.2340790786483337.


Fold 4 IBS: 0.23956009050127555
Fold 5 IBS: 0.2271744056625828
[I 2024-04-16 20:24:00,127] Trial 75 finished with value: 0.2342136389146276 and parameters: {'l1_ratio': 0.06553896583892481}. Best is trial 49 with value: 0.2340790786483337.
Fold 1 IBS: 0.4616056246886181
Fold 2 IBS: 0.22727668657042285
Fold 3 IBS: 0.2287506528643878
Fold 4 IBS: 0.3459040248125855
Fold 5 IBS: 0.3371648911673263
[I 2024-04-16 20:24:00,739] Trial 76 finished with value: 0.3201403760206681 and parameters: {'l1_ratio': 0.1530781934236944}. Best is trial 49 with value: 0.2340790786483337.
Fold 1 IBS: 0.24651237705629012
Fold 2 IBS: 0.2308879732498956
Fold 3 IBS: 0.22891347637323045
Fold 4 IBS: 0.24068462849182948
Fold 5 IBS: 0.22828690062418514
[I 2024-04-16 20:24:00,976] Trial 77 finished with value: 0.23505707115908617 and parameters: {'l1_ratio': 0.02978065555037694}. Best is trial 49 with value: 0.2340790786483337.
Fold 1 IBS: 0.4524340523933975
Fold 2 IBS: 0.22859380602326862
Fold 3 IBS: 0.22879799925262

In [49]:
train_cindex['CoxElastic'] = np.round(study_cindex.best_value, 3)
train_ibs['CoxElastic'] = np.round(study_ibs.best_value, 3)

In [50]:
print("train_cindex: ", np.round(study_cindex.best_value, 3))
print("train_ibs: ", np.round(study_ibs.best_value, 3))

train_cindex:  0.639
train_ibs:  0.234


#### Test

In [51]:
# y into array 
lists = [] 
for i, j in zip(y['event_DFS'], y['DFS']): 
    lists.append((i, j))

y = np.array(lists, dtype=[('status', bool), ('time', np.int32)])

In [52]:
# A function for building the best model with the best parameters 
def create_best_model(model_class, best_params):
    return model_class(**best_params, fit_baseline_model=True)

# Set the best model 
best_model_cindex = create_best_model(CoxnetSurvivalAnalysis, 
                                      study_cindex.best_params)

# Train the best model for C-index on the whole dataset
best_model_cindex.fit(X_new_std, y)

# Evaluate the best model for C-index on MAASTRO dataset
c_index = best_model_cindex.score(MAASTRO_new_std, y_MAASTRO)
c_index = np.round(c_index, 3)
print("test_cindex :", c_index)

# Set the best model 
best_model_ibs = create_best_model(CoxnetSurvivalAnalysis, study_ibs.best_params)

# Train the best model for IBS on the whole dataset
best_model_ibs.fit(X_new_std, y)

# Evaluate the best model for IBS on MAASTRO dataset
lower, upper = np.percentile(y_MAASTRO["time"], [10, 90])
times = np.arange(lower, upper)
surv_prob = np.row_stack([fn(times) for fn in best_model_ibs.predict_survival_function(MAASTRO_new_std)])
ibs = integrated_brier_score(y_MAASTRO, y_MAASTRO, surv_prob, times)
ibs = np.round(ibs, 3)
print("test_ibs: ", ibs)

CoxnetSurvivalAnalysis(fit_baseline_model=True, l1_ratio=0.07142581107906198)

test_cindex : 0.54


CoxnetSurvivalAnalysis(fit_baseline_model=True, l1_ratio=0.07201890462980487)

test_ibs:  0.228


In [53]:
# Saving the values to the dictionary 
test_cindex['CoxElastic'] = c_index
test_ibs['CoxElastic'] = ibs

### 5. Random Survival Forest

#### Train

In [54]:
# Setting the y format 
y = clinical_train[['DFS', 'event_DFS']]

def create_objective(model_class, metric, X, y):
    def objective(trial): 
        # Suggest values for hyperparameters
        min_samples_split = trial.suggest_int("min_samples_split", 2, 20)
        max_leaf_nodes = trial.suggest_int("max_leaf_nodes", 2, 20)
        min_samples_leaf = trial.suggest_int("min_samples_leaf", 1, 20)
        max_depth = trial.suggest_int("max_depth", 1, 20)
        n_estimators = trial.suggest_int("n_estimators", 1, 500)
        oob_score = trial.suggest_categorical("oob_score", [True, False])
        max_samples = trial.suggest_float("max_samples", 0.1, 1.0) 
        max_features = trial.suggest_categorical("max_features", ["auto", "sqrt", "log2", None])
        min_weight_fraction_leaf = trial.suggest_float("min_weight_fraction_leaf", 0.0, 0.5)
        
        # Include warm_start for C-index optimization
        if metric == "c-index":
            warm_start = trial.suggest_categorical("warm_start", [True, False])
        else:
            warm_start = False  # Exclude warm_start for other metrics
        
        # Create and fit survival model 
        model = model_class(min_samples_split=min_samples_split,
                            min_samples_leaf=min_samples_leaf,
                            max_leaf_nodes=max_leaf_nodes,
                            n_estimators=n_estimators, 
                            oob_score=oob_score,
                            warm_start=warm_start,
                            max_depth=max_depth,
                            max_features=max_features,
                            min_weight_fraction_leaf=min_weight_fraction_leaf, 
                            max_samples=max_samples, 
                            random_state=123)

        scores = [] 

        skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=123)
        
        for k, (train_index, test_index) in enumerate(skf.split(X, y.iloc[:, 1])): 
            X_train, X_test = X.iloc[train_index], X.iloc[test_index]
            y_train_df, y_test_df = y.iloc[train_index], y.iloc[test_index]
            
            # y_train into array 
            y_train = [] 
            for i, j in zip(y_train_df['event_DFS'], y_train_df['DFS']): 
                y_train.append((i, j))
            y_train = np.array(y_train, dtype=[('status', bool), ('time', np.int32)])

            # y_test into array
            y_test = [] 
            for i, j in zip(y_test_df['event_DFS'], y_test_df['DFS']): 
                y_test.append((i, j))
            y_test = np.array(y_test, dtype=[('status', bool), ('time', np.int32)])
            
            model.fit(X_train, y_train)

            if metric == "c-index":
                # Make predictions using C-index 
                c_index_score = model.score(X_test, y_test)
                scores.append(c_index_score)
                print(f"Fold {k + 1} C-index: {c_index_score}")
                
            elif metric == "ibs":
                # Make predictions using IBS 
                lower, upper = np.percentile(y_test["time"], [10, 90])    
                times = np.arange(lower, upper)
                cox_surv_prob = np.row_stack([fn(times) for fn in model.predict_survival_function(X_test)])
                ibs = integrated_brier_score(y_test, y_test, cox_surv_prob, times)
                scores.append(ibs)
                print(f"Fold {k + 1} IBS: {ibs}")
            else:
                raise ValueError("Invalid metric. Use 'C-index' or 'ibs'.")
        
        # Return the mean of scores
        return np.mean(scores)
    
    return objective

# C-index
study_cindex = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=123))
objective_cindex = create_objective(RandomSurvivalForest, "c-index", X_new, y)
study_cindex.optimize(objective_cindex, n_trials=100, show_progress_bar=True)
print("\n")
print("* Best trial for C-index: \n", study_cindex.best_trial)
print("\n")
print("* Best Score for C-index: \n", study_cindex.best_value)

# Example usage for IBS
study_ibs = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=123))
objective_ibs = create_objective(RandomSurvivalForest, "ibs", X_new, y)
study_ibs.optimize(objective_ibs, n_trials=100, show_progress_bar=True)
print("\n")
print("* Best trial for IBS: \n", study_ibs.best_trial)
print("\n")
print("* Best Score for IBS: \n", study_ibs.best_value)

[I 2024-04-16 20:24:11,439] A new study created in memory with name: no-name-c9d8190a-30c2-4a22-b725-6ce4a1d62e14


  0%|          | 0/100 [00:00<?, ?it/s]

Fold 1 C-index: 0.6294820717131474
Fold 2 C-index: 0.751937984496124
Fold 3 C-index: 0.6680851063829787
Fold 4 C-index: 0.7376425855513308
Fold 5 C-index: 0.6566523605150214
[I 2024-04-16 20:24:29,571] Trial 0 finished with value: 0.6887600217317204 and parameters: {'min_samples_split': 15, 'max_leaf_nodes': 7, 'min_samples_leaf': 5, 'max_depth': 12, 'n_estimators': 360, 'oob_score': False, 'max_samples': 0.7163467647263769, 'max_features': None, 'min_weight_fraction_leaf': 0.2192861223398122, 'warm_start': False}. Best is trial 0 with value: 0.6887600217317204.
Fold 1 C-index: 0.5418326693227091
Fold 2 C-index: 0.6550387596899225
Fold 3 C-index: 0.7191489361702128
Fold 4 C-index: 0.6349809885931559
Fold 5 C-index: 0.5321888412017167
[I 2024-04-16 20:24:32,663] Trial 1 finished with value: 0.6166380389955435 and parameters: {'min_samples_split': 16, 'max_leaf_nodes': 5, 'min_samples_leaf': 4, 'max_depth': 11, 'n_estimators': 266, 'oob_score': False, 'max_samples': 0.7520097923745717, '

Fold 5 C-index: 0.6330472103004292
[I 2024-04-16 20:25:35,585] Trial 16 finished with value: 0.6103319349924515 and parameters: {'min_samples_split': 18, 'max_leaf_nodes': 2, 'min_samples_leaf': 18, 'max_depth': 6, 'n_estimators': 4, 'oob_score': True, 'max_samples': 0.8500299963488761, 'max_features': 'log2', 'min_weight_fraction_leaf': 0.3855568975046046, 'warm_start': True}. Best is trial 14 with value: 0.7054382711677689.
Fold 1 C-index: 0.49800796812749004
Fold 2 C-index: 0.810077519379845
Fold 3 C-index: 0.851063829787234
Fold 4 C-index: 0.7642585551330798
Fold 5 C-index: 0.7467811158798283
[I 2024-04-16 20:25:36,237] Trial 17 finished with value: 0.7340377976614955 and parameters: {'min_samples_split': 20, 'max_leaf_nodes': 8, 'min_samples_leaf': 16, 'max_depth': 7, 'n_estimators': 96, 'oob_score': True, 'max_samples': 0.9816256083986692, 'max_features': 'log2', 'min_weight_fraction_leaf': 0.10458249310616385, 'warm_start': True}. Best is trial 17 with value: 0.7340377976614955.

Fold 1 C-index: 0.5139442231075697
Fold 2 C-index: 0.8023255813953488
Fold 3 C-index: 0.8723404255319149
Fold 4 C-index: 0.7642585551330798
Fold 5 C-index: 0.7939914163090128
[I 2024-04-16 20:26:00,320] Trial 31 finished with value: 0.7493720402953852 and parameters: {'min_samples_split': 2, 'max_leaf_nodes': 18, 'min_samples_leaf': 8, 'max_depth': 13, 'n_estimators': 239, 'oob_score': True, 'max_samples': 0.6565648650889025, 'max_features': 'sqrt', 'min_weight_fraction_leaf': 0.14061763521617526, 'warm_start': True}. Best is trial 27 with value: 0.7847011373516671.
Fold 1 C-index: 0.5219123505976095
Fold 2 C-index: 0.8062015503875969
Fold 3 C-index: 0.8808510638297873
Fold 4 C-index: 0.8136882129277566
Fold 5 C-index: 0.8454935622317596
[I 2024-04-16 20:26:02,280] Trial 32 finished with value: 0.7736293479949021 and parameters: {'min_samples_split': 5, 'max_leaf_nodes': 17, 'min_samples_leaf': 7, 'max_depth': 12, 'n_estimators': 241, 'oob_score': True, 'max_samples': 0.717337635186249

...

Fold 1 IBS: 0.22149490022552124
Fold 2 IBS: 0.18466865634717505
Fold 3 IBS: 0.23548539432841317
Fold 4 IBS: 0.2113697424267566
Fold 5 IBS: 0.2119677581906565
[I 2024-04-16 20:57:00,338] Trial 32 finished with value: 0.2129972903037045 and parameters: {'min_samples_split': 9, 'max_leaf_nodes': 11, 'min_samples_leaf': 8, 'max_depth': 14, 'n_estimators': 294, 'oob_score': True, 'max_samples': 0.9673759674409311, 'max_features': None, 'min_weight_fraction_leaf': 0.27417727195073677}. Best is trial 32 with value: 0.2129972903037045.
Fold 1 IBS: 0.2567229873354098
Fold 2 IBS: 0.18854079292396503
Fold 3 IBS: 0.2249457788925835
Fold 4 IBS: 0.22508129343804778
Fold 5 IBS: 0.21386415001842546
[I 2024-04-16 20:57:46,311] Trial 33 finished with value: 0.2218310005216863 and parameters: {'min_samples_split': 10, 'max_leaf_nodes': 13, 'min_samples_leaf': 13, 'max_depth': 13, 'n_estimators': 304, 'oob_score': True, 'max_samples': 0.9964442664551478, 'max_features': None, 'min_weight_fraction_leaf': 0

Fold 1 IBS: 0.23233061602902896
Fold 2 IBS: 0.20470776398952695
Fold 3 IBS: 0.24473722549758978
Fold 4 IBS: 0.23554082329183057
Fold 5 IBS: 0.22266798632628182
[I 2024-04-16 21:02:41,797] Trial 48 finished with value: 0.22799688302685164 and parameters: {'min_samples_split': 13, 'max_leaf_nodes': 9, 'min_samples_leaf': 9, 'max_depth': 14, 'n_estimators': 277, 'oob_score': True, 'max_samples': 0.7486071388218823, 'max_features': None, 'min_weight_fraction_leaf': 0.3316489763494004}. Best is trial 43 with value: 0.2123917039278164.
Fold 1 IBS: 0.22140751210407167
Fold 2 IBS: 0.18446093935852728
Fold 3 IBS: 0.23819663142832964
Fold 4 IBS: 0.21304733351095642
Fold 5 IBS: 0.21354093321412873
[I 2024-04-16 21:03:09,885] Trial 49 finished with value: 0.21413066992320276 and parameters: {'min_samples_split': 7, 'max_leaf_nodes': 18, 'min_samples_leaf': 20, 'max_depth': 11, 'n_estimators': 373, 'oob_score': True, 'max_samples': 0.9104681524496127, 'max_features': None, 'min_weight_fraction_leaf

Fold 1 IBS: 0.2365430487443625
Fold 2 IBS: 0.18238280317098843
Fold 3 IBS: 0.2322276544017584
Fold 4 IBS: 0.21380685041046066
Fold 5 IBS: 0.21115528132467798
[I 2024-04-16 21:09:59,603] Trial 64 finished with value: 0.21522312761044962 and parameters: {'min_samples_split': 3, 'max_leaf_nodes': 18, 'min_samples_leaf': 17, 'max_depth': 9, 'n_estimators': 405, 'oob_score': True, 'max_samples': 0.9369908825823273, 'max_features': None, 'min_weight_fraction_leaf': 0.19119487575645203}. Best is trial 43 with value: 0.2123917039278164.
Fold 1 IBS: 0.2545794604503248
Fold 2 IBS: 0.18515187912067008
Fold 3 IBS: 0.22861527937785475
Fold 4 IBS: 0.22050482646274178
Fold 5 IBS: 0.2129134229864224
[I 2024-04-16 21:10:42,663] Trial 65 finished with value: 0.22035297367960277 and parameters: {'min_samples_split': 6, 'max_leaf_nodes': 16, 'min_samples_leaf': 15, 'max_depth': 6, 'n_estimators': 362, 'oob_score': True, 'max_samples': 0.9998359499913109, 'max_features': None, 'min_weight_fraction_leaf': 0

Fold 1 IBS: 0.2511057207255618
Fold 2 IBS: 0.21572593535559278
Fold 3 IBS: 0.2149073004231141
Fold 4 IBS: 0.2472468833843753
Fold 5 IBS: 0.24247425822381166
[I 2024-04-16 21:18:11,915] Trial 80 finished with value: 0.23429201962249113 and parameters: {'min_samples_split': 12, 'max_leaf_nodes': 20, 'min_samples_leaf': 16, 'max_depth': 12, 'n_estimators': 335, 'oob_score': True, 'max_samples': 0.3618038809495957, 'max_features': 'auto', 'min_weight_fraction_leaf': 0.15390327148267055}. Best is trial 43 with value: 0.2123917039278164.
Fold 1 IBS: 0.22731065610063553
Fold 2 IBS: 0.18233470119562936
Fold 3 IBS: 0.23388585350331287
Fold 4 IBS: 0.21305351886000728
Fold 5 IBS: 0.21153134197254575
[I 2024-04-16 21:19:06,532] Trial 81 finished with value: 0.21362321432642614 and parameters: {'min_samples_split': 10, 'max_leaf_nodes': 17, 'min_samples_leaf': 18, 'max_depth': 11, 'n_estimators': 318, 'oob_score': True, 'max_samples': 0.9436804929256887, 'max_features': None, 'min_weight_fraction_l

Fold 1 IBS: 0.21975345965751325
Fold 2 IBS: 0.1826687616539912
Fold 3 IBS: 0.23593232323623828
Fold 4 IBS: 0.21692323202986982
Fold 5 IBS: 0.2157946933853038
[I 2024-04-16 21:31:49,454] Trial 96 finished with value: 0.21421449399258327 and parameters: {'min_samples_split': 8, 'max_leaf_nodes': 9, 'min_samples_leaf': 20, 'max_depth': 12, 'n_estimators': 296, 'oob_score': True, 'max_samples': 0.8401991280225705, 'max_features': None, 'min_weight_fraction_leaf': 0.20291905202477475}. Best is trial 85 with value: 0.21229918040313978.
Fold 1 IBS: 0.22595559817931396
Fold 2 IBS: 0.18811290557691393
Fold 3 IBS: 0.23790732230802805
Fold 4 IBS: 0.22652570350197412
Fold 5 IBS: 0.21929538367243218
[I 2024-04-16 21:32:02,258] Trial 97 finished with value: 0.21955938264773245 and parameters: {'min_samples_split': 7, 'max_leaf_nodes': 7, 'min_samples_leaf': 19, 'max_depth': 15, 'n_estimators': 270, 'oob_score': True, 'max_samples': 0.9001142343235446, 'max_features': None, 'min_weight_fraction_leaf'

In [55]:
train_cindex['Randomsurvivalforest'] = np.round(study_cindex.best_value, 3)
train_ibs['Randomsurvivalforest'] = np.round(study_ibs.best_value, 3)

In [56]:
print("train_cindex: ", np.round(study_cindex.best_value, 3))
print("train_ibs: ", np.round(study_ibs.best_value, 3))

train_cindex:  0.847
train_ibs:  0.212


#### Test

In [57]:
# y into array 
lists = [] 
for i, j in zip(y['event_DFS'], y['DFS']): 
    lists.append((i, j))

In [58]:
y = np.array(lists, dtype=[('status', bool), ('time', np.int32)])
 
# A function for building the best model with the best parameters 
def create_best_model(model_class, best_params):
    best_params["random_state"]=123
    return model_class(**best_params)

# Set the best model 
best_model_cindex = create_best_model(RandomSurvivalForest, study_cindex.best_params)

# Train the best model for C-index on the whole dataset
best_model_cindex.fit(X_new, y)

# Evaluate the best model for C-index on MAASTRO dataset
c_index = best_model_cindex.score(MAASTRO_new, y_MAASTRO)
c_index = np.round(c_index, 3)
print("test_cindex: ", c_index)

# Set the best model 
best_model_ibs = create_best_model(RandomSurvivalForest, study_ibs.best_params)

# Train the best model for IBS on the whole dataset
best_model_ibs.fit(X_new, y)

# Evaluate the best model for IBS on MAASTRO dataset
lower, upper = np.percentile(y_MAASTRO["time"], [10, 90])
times = np.arange(lower, upper)
surv_prob = np.row_stack([fn(times) for fn in best_model_ibs.predict_survival_function(MAASTRO_new)])
ibs = integrated_brier_score(y_MAASTRO, y_MAASTRO, surv_prob, times)
ibs = np.round(ibs, 3)
print("test_ibs: ", ibs)

RandomSurvivalForest(max_depth=16, max_features='auto', max_leaf_nodes=14,
                     max_samples=0.9873533850424658, min_samples_leaf=4,
                     min_samples_split=8,
                     min_weight_fraction_leaf=0.04287202412707603,
                     n_estimators=441, random_state=123, warm_start=True)

test_cindex:  0.54


RandomSurvivalForest(max_depth=13, max_features=None, max_leaf_nodes=11,
                     max_samples=0.9290276329324785, min_samples_leaf=20,
                     min_samples_split=9,
                     min_weight_fraction_leaf=0.20767416366171196,
                     n_estimators=268, oob_score=True, random_state=123)

test_ibs:  0.262


In [59]:
# Saving the values to the dictionary 
test_cindex['Randomsurvivalforest'] = c_index
test_ibs['Randomsurvivalforest'] = ibs

### 6. ExtraSurvivalTrees

#### Train

In [60]:
# Setting the y format 
y = clinical_train[['DFS', 'event_DFS']]

In [61]:
def create_objective(model_class, metric, X, y):
    def objective(trial): 
        # Suggest values for hyperparameters 
        min_samples_split = trial.suggest_int("min_samples_split", 2, 20)
        max_leaf_nodes = trial.suggest_int("max_leaf_nodes", 2, 20)
        min_samples_leaf = trial.suggest_int("min_samples_leaf", 1, 20)
        max_depth = trial.suggest_int("max_depth", 1, 20)
        n_estimators = trial.suggest_int("n_estimators", 1, 500)
        oob_score = trial.suggest_categorical("oob_score", [True, False])
        warm_start = trial.suggest_categorical("warm_start", [True, False])
        max_features = trial.suggest_categorical("max_features", ["auto", "sqrt", "log2", None, 0.1, 1])
        max_samples = trial.suggest_float("max_samples", 0.1, 1.0) 
        min_weight_fraction_leaf = trial.suggest_float("min_weight_fraction_leaf", 0.0, 0.5)
        
        # Include warm_start for C-index optimization
        if metric == "c-index":
            warm_start = trial.suggest_categorical("warm_start", [True, False])
        else:
            warm_start = False  # Exclude warm_start for other metrics

        
        # Create and fit survival model 
        model = model_class(min_samples_split=min_samples_split,
                            min_samples_leaf=min_samples_leaf,
                            max_leaf_nodes=max_leaf_nodes,
                            n_estimators=n_estimators, 
                            oob_score=oob_score, 
                            max_features=max_features, 
                            warm_start=warm_start, 
                            max_samples=max_samples,
                            min_weight_fraction_leaf=min_weight_fraction_leaf, 
                            max_depth=max_depth, 
                            random_state=123) 
        
        scores = [] 
        
        skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=123)
        
        for k, (train_index, test_index) in enumerate(skf.split(X, y.iloc[:, 1])): 
            X_train, X_test = X.iloc[train_index], X.iloc[test_index]
            y_train_df, y_test_df = y.iloc[train_index], y.iloc[test_index]
            
            # y_train into array 
            y_train = [] 
            for i, j in zip(y_train_df['event_DFS'], y_train_df['DFS']): 
                y_train.append((i, j))
            y_train = np.array(y_train, dtype=[('status', bool), ('time', np.int32)])

            # y_test into array
            y_test = [] 
            for i, j in zip(y_test_df['event_DFS'], y_test_df['DFS']): 
                y_test.append((i, j))
            y_test = np.array(y_test, dtype=[('status', bool), ('time', np.int32)])

            model.fit(X_train, y_train)

            if metric == "c-index":
                # Make predictions using C-index 
                c_index_score = model.score(X_test, y_test)
                scores.append(c_index_score)
                print(f"Fold {k + 1} C-index: {c_index_score}")
                
            elif metric == "ibs":
                # Make predictions using IBS 
                lower, upper = np.percentile(y_test["time"], [10, 90])    
                times = np.arange(lower, upper)
                surv_prob = np.row_stack([fn(times) for fn in model.predict_survival_function(X_test)])
                ibs = integrated_brier_score(y_test, y_test, surv_prob, times)
                scores.append(ibs)
                print(f"Fold {k + 1} IBS: {ibs}")
            else:
                raise ValueError("Invalid metric. Use 'C-index' or 'ibs'.")
        
        # Return the mean of scores
        return np.mean(scores)
    
    return objective

# C-index
study_cindex = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=123))
objective_cindex = create_objective(ExtraSurvivalTrees, "c-index", X_new, y)
study_cindex.optimize(objective_cindex, n_trials=100, show_progress_bar=True)
print("\n")
print("* Best trial for C-index: \n", study_cindex.best_trial)
print("\n")
print("* Best Score for C-index: \n", study_cindex.best_value)

# Example usage for IBS
study_ibs = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=123))
objective_ibs = create_objective(ExtraSurvivalTrees, "ibs", X_new, y)
study_ibs.optimize(objective_ibs, n_trials=100, show_progress_bar=True)
print("\n")
print("* Best trial for IBS: \n", study_ibs.best_trial)
print("\n")
print("* Best Score for IBS: \n", study_ibs.best_value)


[I 2024-04-16 21:32:37,264] A new study created in memory with name: no-name-2decf87a-bcac-44d3-b6a8-88cbceddc6f6


  0%|          | 0/100 [00:00<?, ?it/s]

Fold 1 C-index: 0.50199203187251
Fold 2 C-index: 0.7635658914728682
Fold 3 C-index: 0.8425531914893617
Fold 4 C-index: 0.7376425855513308
Fold 5 C-index: 0.7381974248927039
[I 2024-04-16 21:32:38,122] Trial 0 finished with value: 0.716790225055755 and parameters: {'min_samples_split': 15, 'max_leaf_nodes': 7, 'min_samples_leaf': 5, 'max_depth': 12, 'n_estimators': 360, 'oob_score': False, 'warm_start': True, 'max_features': 'log2', 'max_samples': 0.7641958651588321, 'min_weight_fraction_leaf': 0.09124586522674999}. Best is trial 0 with value: 0.716790225055755.
Fold 1 C-index: 0.5
Fold 2 C-index: 0.5
Fold 3 C-index: 0.5
Fold 4 C-index: 0.5
Fold 5 C-index: 0.5
[I 2024-04-16 21:32:40,048] Trial 1 finished with value: 0.5 and parameters: {'min_samples_split': 5, 'max_leaf_nodes': 12, 'min_samples_leaf': 11, 'max_depth': 13, 'n_estimators': 425, 'oob_score': True, 'warm_start': True, 'max_features': None, 'max_samples': 0.4877764869966794, 'min_weight_fraction_leaf': 0.2468425488251531}. B

Fold 1 C-index: 0.5
Fold 2 C-index: 0.5
Fold 3 C-index: 0.5
Fold 4 C-index: 0.5
Fold 5 C-index: 0.5
[I 2024-04-16 21:33:02,543] Trial 16 finished with value: 0.5 and parameters: {'min_samples_split': 20, 'max_leaf_nodes': 14, 'min_samples_leaf': 20, 'max_depth': 4, 'n_estimators': 231, 'oob_score': False, 'warm_start': True, 'max_features': None, 'max_samples': 0.6778661833429002, 'min_weight_fraction_leaf': 0.35612645124522524}. Best is trial 15 with value: 0.7572849484519277.
Fold 1 C-index: 0.5697211155378487
Fold 2 C-index: 0.7209302325581395
Fold 3 C-index: 0.8170212765957446
Fold 4 C-index: 0.7224334600760456
Fold 5 C-index: 0.6995708154506438
[I 2024-04-16 21:33:03,033] Trial 17 finished with value: 0.7059353800436845 and parameters: {'min_samples_split': 17, 'max_leaf_nodes': 11, 'min_samples_leaf': 14, 'max_depth': 9, 'n_estimators': 100, 'oob_score': False, 'warm_start': True, 'max_features': None, 'max_samples': 0.3861349103797888, 'min_weight_fraction_leaf': 0.0896057534547

Fold 1 C-index: 0.6055776892430279
Fold 2 C-index: 0.7635658914728682
Fold 3 C-index: 0.8723404255319149
Fold 4 C-index: 0.7414448669201521
Fold 5 C-index: 0.7424892703862661
[I 2024-04-16 21:33:14,250] Trial 31 finished with value: 0.7450836287108459 and parameters: {'min_samples_split': 9, 'max_leaf_nodes': 11, 'min_samples_leaf': 8, 'max_depth': 1, 'n_estimators': 89, 'oob_score': False, 'warm_start': True, 'max_features': None, 'max_samples': 0.6115041201105692, 'min_weight_fraction_leaf': 0.07529457384788989}. Best is trial 24 with value: 0.7999244832738667.
Fold 1 C-index: 0.6135458167330677
Fold 2 C-index: 0.8217054263565892
Fold 3 C-index: 0.8765957446808511
Fold 4 C-index: 0.8098859315589354
Fold 5 C-index: 0.8240343347639485
[I 2024-04-16 21:33:14,710] Trial 32 finished with value: 0.7891534508186784 and parameters: {'min_samples_split': 12, 'max_leaf_nodes': 12, 'min_samples_leaf': 6, 'max_depth': 2, 'n_estimators': 106, 'oob_score': False, 'warm_start': True, 'max_features'

...

Fold 1 IBS: 0.24189951983836425
Fold 2 IBS: 0.22116291305449884
Fold 3 IBS: 0.21713072638394984
Fold 4 IBS: 0.24023416048014104
Fold 5 IBS: 0.22691624187044754
[I 2024-04-16 21:36:50,939] Trial 30 finished with value: 0.2294687123254803 and parameters: {'min_samples_split': 13, 'max_leaf_nodes': 8, 'min_samples_leaf': 7, 'max_depth': 5, 'n_estimators': 180, 'oob_score': False, 'warm_start': True, 'max_features': 'sqrt', 'max_samples': 0.5988223393644435, 'min_weight_fraction_leaf': 0.20241101766143602}. Best is trial 15 with value: 0.21555098369824366.
Fold 1 IBS: 0.24364620279536037
Fold 2 IBS: 0.2205505651382432
Fold 3 IBS: 0.21184589055027794
Fold 4 IBS: 0.24142198209294274
Fold 5 IBS: 0.2268782233658812
[I 2024-04-16 21:36:52,784] Trial 31 finished with value: 0.22886857278854106 and parameters: {'min_samples_split': 17, 'max_leaf_nodes': 13, 'min_samples_leaf': 11, 'max_depth': 8, 'n_estimators': 162, 'oob_score': False, 'warm_start': True, 'max_features': 'auto', 'max_samples': 0

Fold 1 IBS: 0.24970180838617465
Fold 2 IBS: 0.20711640796461644
Fold 3 IBS: 0.210622445605441
Fold 4 IBS: 0.2398038048480638
Fold 5 IBS: 0.23687430367329662
[I 2024-04-16 21:38:21,534] Trial 45 finished with value: 0.22882375409551847 and parameters: {'min_samples_split': 6, 'max_leaf_nodes': 19, 'min_samples_leaf': 6, 'max_depth': 13, 'n_estimators': 481, 'oob_score': True, 'warm_start': False, 'max_features': 'sqrt', 'max_samples': 0.6241818506764952, 'min_weight_fraction_leaf': 0.003649071529300437}. Best is trial 41 with value: 0.2125643090401542.
Fold 1 IBS: 0.23201805390304453
Fold 2 IBS: 0.20002733588165444
Fold 3 IBS: 0.19908476397700547
Fold 4 IBS: 0.22074033459452136
Fold 5 IBS: 0.21494542822340232
[I 2024-04-16 21:38:30,399] Trial 46 finished with value: 0.2133631833159256 and parameters: {'min_samples_split': 4, 'max_leaf_nodes': 17, 'min_samples_leaf': 2, 'max_depth': 15, 'n_estimators': 419, 'oob_score': True, 'warm_start': False, 'max_features': None, 'max_samples': 0.55

Fold 1 IBS: 0.24516656524539981
Fold 2 IBS: 0.21062905264741455
Fold 3 IBS: 0.20928713648563754
Fold 4 IBS: 0.24177268946133224
Fold 5 IBS: 0.23349939521162735
[I 2024-04-16 21:40:12,379] Trial 60 finished with value: 0.2280709678102823 and parameters: {'min_samples_split': 7, 'max_leaf_nodes': 13, 'min_samples_leaf': 10, 'max_depth': 12, 'n_estimators': 382, 'oob_score': True, 'warm_start': False, 'max_features': 'sqrt', 'max_samples': 0.7740796078428375, 'min_weight_fraction_leaf': 0.004254061138558238}. Best is trial 59 with value: 0.21106902851963083.
Fold 1 IBS: 0.23239287585025117
Fold 2 IBS: 0.19481155216054105
Fold 3 IBS: 0.19221809734433284
Fold 4 IBS: 0.21072513374212
Fold 5 IBS: 0.21430420045958357
[I 2024-04-16 21:40:23,847] Trial 61 finished with value: 0.20889037191136572 and parameters: {'min_samples_split': 7, 'max_leaf_nodes': 14, 'min_samples_leaf': 9, 'max_depth': 12, 'n_estimators': 258, 'oob_score': True, 'warm_start': False, 'max_features': None, 'max_samples': 0.

Fold 1 IBS: 0.23374241288259937
Fold 2 IBS: 0.19930439672774886
Fold 3 IBS: 0.18717738406829976
Fold 4 IBS: 0.20823176235963525
Fold 5 IBS: 0.21501371316102338
[I 2024-04-16 21:43:16,129] Trial 75 finished with value: 0.20869393383986132 and parameters: {'min_samples_split': 5, 'max_leaf_nodes': 11, 'min_samples_leaf': 10, 'max_depth': 12, 'n_estimators': 435, 'oob_score': True, 'warm_start': False, 'max_features': None, 'max_samples': 0.9528867121166885, 'min_weight_fraction_leaf': 0.0012590377645196535}. Best is trial 73 with value: 0.20839397477875155.
Fold 1 IBS: 0.24525891729558977
Fold 2 IBS: 0.21896003468791347
Fold 3 IBS: 0.21598508493564658
Fold 4 IBS: 0.24394612844374725
Fold 5 IBS: 0.2322350643210711
[I 2024-04-16 21:43:23,448] Trial 76 finished with value: 0.23127704593679366 and parameters: {'min_samples_split': 5, 'max_leaf_nodes': 11, 'min_samples_leaf': 13, 'max_depth': 11, 'n_estimators': 465, 'oob_score': True, 'warm_start': False, 'max_features': 'log2', 'max_samples

Fold 1 IBS: 0.24308229605873613
Fold 2 IBS: 0.20243085243083073
Fold 3 IBS: 0.20257300733372727
Fold 4 IBS: 0.2350012477048381
Fold 5 IBS: 0.22812847237666628
[I 2024-04-16 21:45:32,412] Trial 90 finished with value: 0.2222431751809597 and parameters: {'min_samples_split': 3, 'max_leaf_nodes': 7, 'min_samples_leaf': 9, 'max_depth': 14, 'n_estimators': 427, 'oob_score': True, 'warm_start': False, 'max_features': 0.1, 'max_samples': 0.9384169100812306, 'min_weight_fraction_leaf': 0.10055258156251395}. Best is trial 82 with value: 0.20797277264373898.
Fold 1 IBS: 0.23494236000327381
Fold 2 IBS: 0.19851393414446025
Fold 3 IBS: 0.188316054669442
Fold 4 IBS: 0.20779679335240808
Fold 5 IBS: 0.21280969619087245
[I 2024-04-16 21:45:42,579] Trial 91 finished with value: 0.20847576767209133 and parameters: {'min_samples_split': 2, 'max_leaf_nodes': 8, 'min_samples_leaf': 10, 'max_depth': 12, 'n_estimators': 442, 'oob_score': True, 'warm_start': False, 'max_features': None, 'max_samples': 0.961995

In [62]:
train_cindex['ExtraSurvivalTrees'] = np.round(study_cindex.best_value, 3)
train_ibs['ExtraSurvivalTrees'] = np.round(study_ibs.best_value, 3)

In [63]:
print("train_cindex: ", np.round(study_cindex.best_value, 3))
print("train_ibs: ", np.round(study_ibs.best_value, 3))

train_cindex:  0.842
train_ibs:  0.208


#### Test

In [64]:
# y into array 
lists = [] 
for i, j in zip(y['event_DFS'], y['DFS']): 
    lists.append((i, j))

y = np.array(lists, dtype=[('status', bool), ('time', np.int32)])

In [65]:
# A function for building the best model with the best parameters 
def create_best_model(model_class, best_params):
    best_params["random_state"]=123
    return model_class(**best_params)

# Set the best model 
best_model_cindex = create_best_model(ExtraSurvivalTrees, study_cindex.best_params)

# Train the best model for C-index on the whole dataset
best_model_cindex.fit(X_new, y)

# Evaluate the best model for C-index on MAASTRO dataset
c_index = best_model_cindex.score(MAASTRO_new, y_MAASTRO)
c_index = np.round(c_index, 3)
print("C-index score:", c_index)

# Set the best model 
best_model_ibs = create_best_model(ExtraSurvivalTrees, study_ibs.best_params)

# Train the best model for IBS on the whole dataset
best_model_ibs.fit(X_new, y)

# Evaluate the best model for IBS on MAASTRO dataset
lower, upper = np.percentile(y_MAASTRO["time"], [10, 90])
times = np.arange(lower, upper)
surv_prob = np.row_stack([fn(times) for fn in best_model_ibs.predict_survival_function(MAASTRO_new)])
ibs = integrated_brier_score(y_MAASTRO, y_MAASTRO, surv_prob, times)
ibs = np.round(ibs, 3)
print("IBS:", ibs)

ExtraSurvivalTrees(max_depth=16, max_features=None, max_leaf_nodes=10,
                   max_samples=0.9186714469202719, min_samples_leaf=2,
                   min_samples_split=3,
                   min_weight_fraction_leaf=0.04956115599708024,
                   n_estimators=474, random_state=123, warm_start=True)

C-index score: 0.567


ExtraSurvivalTrees(max_depth=12, max_features=None, max_leaf_nodes=9,
                   max_samples=0.9407754324156289, min_samples_leaf=9,
                   min_samples_split=2,
                   min_weight_fraction_leaf=0.04206241234084829,
                   n_estimators=431, oob_score=True, random_state=123)

IBS: 0.235


In [66]:
# Saving the values to the dictionary 
test_cindex['ExtraSurvivalTrees'] = c_index
test_ibs['ExtraSurvivalTrees'] = ibs

### 7. GradientBoostingSurvivalAnalysis

#### Train

In [67]:
# Setting the y format 
y = clinical_train[['DFS', 'event_DFS']]

# Running to optuna for hyperparameter tuning
def create_objective(model_class, metric, X, y):
    def objective(trial): 
        # Suggest values for hyperparameters
        subsample = trial.suggest_float("subsample", 0.1, 1)
        learning_rate = trial.suggest_float("learning_rate", 0.001, 0.1)
        dropout_rate = trial.suggest_float("dropout_rate", 0.1, 1)
        n_estimators = trial.suggest_int("n_estimators", 1, 500)
        criterion = trial.suggest_categorical('criterion', ['friedman_mse', 'squared_error'])
        ccp_alpha = trial.suggest_float("ccp_alpha", 0.0, 10)
        min_weight_fraction_leaf = trial.suggest_float("min_weight_fraction_leaf", 0.0, 0.5)
        max_features = trial.suggest_categorical("max_features", ["auto", "sqrt", "log2", None, 0.1, 1])
        min_impurity_decrease = trial.suggest_loguniform('min_impurity_decrease', 1e-7, 1e-1)
        validation_fraction = trial.suggest_float("validation_fraction", 0.0, 1.0)
        min_samples_split = trial.suggest_int("min_samples_split", 2, 20)
        max_leaf_nodes = trial.suggest_int("max_leaf_nodes", 2, 20)
        min_samples_leaf = trial.suggest_int("min_samples_leaf", 1, 20)
        max_depth = trial.suggest_int("max_depth", 1, 20)
        
        # Create and fit survival model 
        model = model_class(subsample=subsample,
                            learning_rate=learning_rate,
                            dropout_rate=dropout_rate,
                            n_estimators=n_estimators,
                            ccp_alpha=ccp_alpha, 
                            criterion=criterion,
                            min_samples_split=min_samples_split,
                            min_samples_leaf=min_samples_leaf,
                            min_weight_fraction_leaf=min_weight_fraction_leaf,
                            max_depth=max_depth,
                            max_features=max_features,
                            max_leaf_nodes=max_leaf_nodes, 
                            min_impurity_decrease=min_impurity_decrease,
                            validation_fraction=validation_fraction, 
                            random_state=123)
        
        scores = [] 
        
        skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=123)
        
        for k, (train_index, test_index) in enumerate(skf.split(X, y.iloc[:, 1])): 
            X_train, X_test = X.iloc[train_index], X.iloc[test_index]
            y_train_df, y_test_df = y.iloc[train_index], y.iloc[test_index]
            
            # y_train into array 
            y_train = [] 
            for i, j in zip(y_train_df['event_DFS'], y_train_df['DFS']): 
                y_train.append((i, j))
            y_train = np.array(y_train, dtype=[('status', bool), ('time', np.int32)])

            # y_test into array
            y_test = [] 
            for i, j in zip(y_test_df['event_DFS'], y_test_df['DFS']): 
                y_test.append((i, j))
            y_test = np.array(y_test, dtype=[('status', bool), ('time', np.int32)])

            model.fit(X_train, y_train)

            if metric == "c-index":
                # Make predictions using C-index 
                c_index_score = model.score(X_test, y_test)
                scores.append(c_index_score)
                print(f"Fold {k + 1} C-index: {c_index_score}")
                
            elif metric == "ibs":
                # Make predictions using IBS 
                lower, upper = np.percentile(y_test["time"], [10, 90])    
                times = np.arange(lower, upper)
                surv_prob = np.row_stack([fn(times) for fn in model.predict_survival_function(X_test)])
                ibs = integrated_brier_score(y_test, y_test, surv_prob, times)
                scores.append(ibs)
                print(f"Fold {k + 1} IBS: {ibs}")
            else:
                raise ValueError("Invalid metric. Use 'C-index' or 'ibs'.")
        
        # Return the mean of scores
        return np.mean(scores)
    
    return objective

# C-index
study_cindex = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=123))
objective_cindex = create_objective(GradientBoostingSurvivalAnalysis, "c-index", X_new, y)
study_cindex.optimize(objective_cindex, n_trials=100, show_progress_bar=True)
print("\n")
print("* Best trial for C-index: \n", study_cindex.best_trial)
print("\n")
print("* Best Score for C-index: \n", study_cindex.best_value)

# Example usage for IBS
study_ibs = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=123))
objective_ibs = create_objective(GradientBoostingSurvivalAnalysis, "ibs", X_new, y)
study_ibs.optimize(objective_ibs, n_trials=100, show_progress_bar=True)
print("\n")
print("* Best trial for IBS: \n", study_ibs.best_trial)
print("\n")
print("* Best Score for IBS: \n", study_ibs.best_value)

[I 2024-04-16 21:47:08,809] A new study created in memory with name: no-name-ec4a3edf-ba81-48f5-9b10-098112766976


  0%|          | 0/100 [00:00<?, ?it/s]

Fold 1 C-index: 0.5
Fold 2 C-index: 0.5
Fold 3 C-index: 0.5
Fold 4 C-index: 0.5
Fold 5 C-index: 0.5
[I 2024-04-16 21:47:36,274] Trial 0 finished with value: 0.5 and parameters: {'subsample': 0.7268222670380755, 'learning_rate': 0.02932779416008757, 'dropout_rate': 0.3041663082077828, 'n_estimators': 276, 'criterion': 'friedman_mse', 'ccp_alpha': 9.807641983846155, 'min_weight_fraction_leaf': 0.34241486929243165, 'max_features': None, 'min_impurity_decrease': 2.4449249473284515e-05, 'validation_fraction': 0.7379954057320357, 'min_samples_split': 5, 'max_leaf_nodes': 5, 'min_samples_leaf': 11, 'max_depth': 11}. Best is trial 0 with value: 0.5.
Fold 1 C-index: 0.5
Fold 2 C-index: 0.5
Fold 3 C-index: 0.5
Fold 4 C-index: 0.5
Fold 5 C-index: 0.5
[I 2024-04-16 21:47:49,337] Trial 1 finished with value: 0.5 and parameters: {'subsample': 0.6709608626961889, 'learning_rate': 0.08509374761370117, 'dropout_rate': 0.7520097923745717, 'n_estimators': 306, 'criterion': 'friedman_mse', 'ccp_alpha': 3.

Fold 1 C-index: 0.5
Fold 2 C-index: 0.5
Fold 3 C-index: 0.5
Fold 4 C-index: 0.5
Fold 5 C-index: 0.5
[I 2024-04-16 21:56:29,079] Trial 13 finished with value: 0.5 and parameters: {'subsample': 0.8386796524426539, 'learning_rate': 0.046734492485875676, 'dropout_rate': 0.4821375662037144, 'n_estimators': 402, 'criterion': 'squared_error', 'ccp_alpha': 1.5696007313501796, 'min_weight_fraction_leaf': 0.18684147934268416, 'max_features': 'log2', 'min_impurity_decrease': 1.5044881127471587e-06, 'validation_fraction': 0.8166356053932342, 'min_samples_split': 16, 'max_leaf_nodes': 19, 'min_samples_leaf': 16, 'max_depth': 4}. Best is trial 12 with value: 0.6841431930275611.
Fold 1 C-index: 0.5
Fold 2 C-index: 0.5
Fold 3 C-index: 0.5
Fold 4 C-index: 0.5
Fold 5 C-index: 0.5
[I 2024-04-16 21:57:23,188] Trial 14 finished with value: 0.5 and parameters: {'subsample': 0.33235389014851724, 'learning_rate': 0.04522573411670834, 'dropout_rate': 0.2712811374536856, 'n_estimators': 405, 'criterion': 'squar

Fold 1 C-index: 0.5
Fold 2 C-index: 0.5
Fold 3 C-index: 0.5
Fold 4 C-index: 0.5
Fold 5 C-index: 0.5
[I 2024-04-16 22:08:07,855] Trial 25 finished with value: 0.5 and parameters: {'subsample': 0.8502968404151126, 'learning_rate': 0.024443951259730985, 'dropout_rate': 0.2675273969344081, 'n_estimators': 441, 'criterion': 'squared_error', 'ccp_alpha': 2.0753749717266823, 'min_weight_fraction_leaf': 0.3503497788125578, 'max_features': 'auto', 'min_impurity_decrease': 7.237153572123947e-07, 'validation_fraction': 0.41222573804914475, 'min_samples_split': 16, 'max_leaf_nodes': 13, 'min_samples_leaf': 12, 'max_depth': 3}. Best is trial 12 with value: 0.6841431930275611.
Fold 1 C-index: 0.5
Fold 2 C-index: 0.5
Fold 3 C-index: 0.5
Fold 4 C-index: 0.5
Fold 5 C-index: 0.5
[I 2024-04-16 22:08:31,887] Trial 26 finished with value: 0.5 and parameters: {'subsample': 0.7670085127536703, 'learning_rate': 0.011828778593594373, 'dropout_rate': 0.4300954216773497, 'n_estimators': 330, 'criterion': 'friedm

...

Fold 3 IBS: 0.22898186806977705
Fold 4 IBS: 0.24197477145927118
Fold 5 IBS: 0.22939559304809248
[I 2024-04-16 23:41:12,145] Trial 55 finished with value: 0.23592784351233073 and parameters: {'subsample': 0.809323041898869, 'learning_rate': 0.0040826970179266685, 'dropout_rate': 0.36065535241543395, 'n_estimators': 435, 'criterion': 'squared_error', 'ccp_alpha': 1.5126136571072866, 'min_weight_fraction_leaf': 0.2002957617254777, 'max_features': 'auto', 'min_impurity_decrease': 2.0868015436948727e-05, 'validation_fraction': 0.8814649292460881, 'min_samples_split': 17, 'max_leaf_nodes': 18, 'min_samples_leaf': 11, 'max_depth': 2}. Best is trial 41 with value: 0.23455338278002208.
Fold 1 IBS: 0.24724710044658998
Fold 2 IBS: 0.23203988453792299
Fold 3 IBS: 0.22898186806977705
Fold 4 IBS: 0.24197477145927113
Fold 5 IBS: 0.22939559304809248
[I 2024-04-16 23:41:31,898] Trial 56 finished with value: 0.23592784351233073 and parameters: {'subsample': 0.2818850250060197, 'learning_rate': 0.0985092

Fold 2 IBS: 0.23203988453792299
Fold 3 IBS: 0.22898186806977708
Fold 4 IBS: 0.24197477145927118
Fold 5 IBS: 0.2293955930480925
[I 2024-04-16 23:45:37,264] Trial 66 finished with value: 0.23592784351233073 and parameters: {'subsample': 0.8489175059797514, 'learning_rate': 0.008448226967076113, 'dropout_rate': 0.12725750793823018, 'n_estimators': 424, 'criterion': 'squared_error', 'ccp_alpha': 0.893877768639663, 'min_weight_fraction_leaf': 0.2702972473075093, 'max_features': 'auto', 'min_impurity_decrease': 7.586810829424823e-05, 'validation_fraction': 0.8971088342813041, 'min_samples_split': 17, 'max_leaf_nodes': 17, 'min_samples_leaf': 8, 'max_depth': 4}. Best is trial 62 with value: 0.23436365650963703.
Fold 1 IBS: 0.24724710044658996
Fold 2 IBS: 0.23203988453792296
Fold 3 IBS: 0.22898186806977708
Fold 4 IBS: 0.24197477145927113
Fold 5 IBS: 0.2293955930480925
[I 2024-04-16 23:45:58,562] Trial 67 finished with value: 0.23592784351233073 and parameters: {'subsample': 0.9594402941587806,

Fold 2 IBS: 0.23203988453792299
Fold 3 IBS: 0.22898186806977705
Fold 4 IBS: 0.24197477145927113
Fold 5 IBS: 0.22939559304809248
[I 2024-04-16 23:49:40,186] Trial 77 finished with value: 0.23592784351233073 and parameters: {'subsample': 0.7234820637720293, 'learning_rate': 0.04128077713454106, 'dropout_rate': 0.19884799526452568, 'n_estimators': 302, 'criterion': 'squared_error', 'ccp_alpha': 1.2832467794713243, 'min_weight_fraction_leaf': 0.4027406894595495, 'max_features': None, 'min_impurity_decrease': 5.35591590419612e-07, 'validation_fraction': 0.8758106078120852, 'min_samples_split': 19, 'max_leaf_nodes': 5, 'min_samples_leaf': 10, 'max_depth': 2}. Best is trial 70 with value: 0.23410964446224067.
Fold 1 IBS: 0.24724710044658998
Fold 2 IBS: 0.23203988453792299
Fold 3 IBS: 0.22898186806977705
Fold 4 IBS: 0.24197477145927113
Fold 5 IBS: 0.2293955930480925
[I 2024-04-16 23:49:56,603] Trial 78 finished with value: 0.23592784351233073 and parameters: {'subsample': 0.6708593402641205, '

Fold 3 IBS: 0.22898186806977705
Fold 4 IBS: 0.24197477145927113
Fold 5 IBS: 0.22939559304809248
[I 2024-04-16 23:52:44,055] Trial 88 finished with value: 0.23592784351233073 and parameters: {'subsample': 0.6879348852579991, 'learning_rate': 0.08209035744510738, 'dropout_rate': 0.24522201083374043, 'n_estimators': 84, 'criterion': 'squared_error', 'ccp_alpha': 0.2983993026610602, 'min_weight_fraction_leaf': 0.3774672254272644, 'max_features': None, 'min_impurity_decrease': 5.4011226362110275e-06, 'validation_fraction': 0.568843188657281, 'min_samples_split': 16, 'max_leaf_nodes': 6, 'min_samples_leaf': 4, 'max_depth': 11}. Best is trial 81 with value: 0.23062671367871226.
Fold 1 IBS: 0.24724710044658998
Fold 2 IBS: 0.23203988453792299
Fold 3 IBS: 0.22898186806977708
Fold 4 IBS: 0.24197477145927113
Fold 5 IBS: 0.2293955930480925
[I 2024-04-16 23:52:47,573] Trial 89 finished with value: 0.23592784351233073 and parameters: {'subsample': 0.6555693108457097, 'learning_rate': 0.08348043724232

Fold 4 IBS: 0.24197477145927113
Fold 5 IBS: 0.2293955930480925
[I 2024-04-16 23:54:59,506] Trial 99 finished with value: 0.23592784351233073 and parameters: {'subsample': 0.4750010457704055, 'learning_rate': 0.07813501740629669, 'dropout_rate': 0.30318261654818046, 'n_estimators': 242, 'criterion': 'squared_error', 'ccp_alpha': 1.7079733343503498, 'min_weight_fraction_leaf': 0.32269779176244495, 'max_features': 0.1, 'min_impurity_decrease': 1.276230329676773e-06, 'validation_fraction': 0.36954438153697, 'min_samples_split': 11, 'max_leaf_nodes': 9, 'min_samples_leaf': 7, 'max_depth': 2}. Best is trial 91 with value: 0.2300644165494142.


* Best trial for IBS: 
 FrozenTrial(number=91, state=TrialState.COMPLETE, values=[0.2300644165494142], datetime_start=datetime.datetime(2024, 4, 16, 23, 52, 53, 434375), datetime_complete=datetime.datetime(2024, 4, 16, 23, 53, 15, 253503), params={'subsample': 0.579213280645811, 'learning_rate': 0.09426091382038485, 'dropout_rate': 0.18479264516541616,

In [68]:
train_cindex['GradientBoosting'] = np.round(study_cindex.best_value, 3)
train_ibs['GradientBoosting'] = np.round(study_ibs.best_value, 3)

In [69]:
print("train_cindex: ", np.round(study_cindex.best_value, 3))
print("train_ibs: ", np.round(study_ibs.best_value, 3))

train_cindex:  0.7
train_ibs:  0.23


#### Test

In [70]:
# y into array 
lists = [] 
for i, j in zip(y['event_DFS'], y['DFS']): 
    lists.append((i, j))

y = np.array(lists, dtype=[('status', bool), ('time', np.int32)])

In [71]:
# A function for building the best model with the best parameters 
def create_best_model(model_class, best_params):
    best_params["random_state"]=123
    return model_class(**best_params)

# Set the best model 
best_model_cindex = create_best_model(GradientBoostingSurvivalAnalysis, study_cindex.best_params)

# Train the best model for C-index on the whole dataset
best_model_cindex.fit(X_new, y)

# Evaluate the best model for C-index on MAASTRO dataset
c_index = best_model_cindex.score(MAASTRO_new, y_MAASTRO)
c_index = np.round(c_index, 3)
print("C-index score:", c_index)

# Set the best model 
best_model_ibs = create_best_model(GradientBoostingSurvivalAnalysis, study_ibs.best_params)

# Train the best model for IBS on the whole dataset
best_model_ibs.fit(X_new, y)

# Evaluate the best model for IBS on MAASTRO dataset
lower, upper = np.percentile(y_MAASTRO["time"], [10, 90])
times = np.arange(lower, upper)
surv_prob = np.row_stack([fn(times) for fn in best_model_ibs.predict_survival_function(MAASTRO_new)])
ibs = integrated_brier_score(y_MAASTRO, y_MAASTRO, surv_prob, times)
ibs = np.round(ibs, 3)
print("IBS:", ibs)

GradientBoostingSurvivalAnalysis(ccp_alpha=0.004507507039685693,
                                 criterion='squared_error',
                                 dropout_rate=0.8022681311855436,
                                 learning_rate=0.010888929954369636,
                                 max_depth=12, max_features='auto',
                                 max_leaf_nodes=3,
                                 min_impurity_decrease=4.0700060224631717e-07,
                                 min_samples_leaf=10, min_samples_split=19,
                                 min_weight_fraction_leaf=0.1685851039915477,
                                 n_estimators=443, random_state=123,
                                 subsample=0.8597838031871279,
                                 validation_fraction=0.8827408742810589)

C-index score: 0.516


GradientBoostingSurvivalAnalysis(ccp_alpha=0.03787089343697578,
                                 criterion='squared_error',
                                 dropout_rate=0.18479264516541616,
                                 learning_rate=0.09426091382038485, max_depth=2,
                                 max_leaf_nodes=6,
                                 min_impurity_decrease=7.4293259533530295e-06,
                                 min_samples_leaf=7, min_samples_split=18,
                                 min_weight_fraction_leaf=0.4036747584607865,
                                 n_estimators=390, random_state=123,
                                 subsample=0.579213280645811,
                                 validation_fraction=0.5325834925366492)

IBS: 0.228


In [72]:
# Saving the values to the dictionary 
test_cindex['GradientBoosting'] = c_index
test_ibs['GradientBoosting'] = ibs

### 8. ComponentwiseGradientBoostingSurvivalAnalysis

#### Train

In [73]:
# Setting the y format 
y = clinical_train[['DFS', 'event_DFS']]

In [74]:
def create_objective(model_class, metric, X, y):
    def objective(trial): 
        # Suggest values for hyperparameters
        subsample = trial.suggest_float("subsample", 0.1, 1)
        dropout_rate = trial.suggest_float("dropout_rate", 0.1, 1)
        n_estimators = trial.suggest_int("n_estimators", 1, 500)
        learning_rate = trial.suggest_float("learning_rate", 0.001, 0.1)
        
        # Create and fit survival model 
        model = model_class(subsample=subsample,
                            dropout_rate=dropout_rate,
                            n_estimators=n_estimators,
                            learning_rate=learning_rate,
                            random_state=123)
                
        scores = [] 
        
        skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=123)
        
        for k, (train_index, test_index) in enumerate(skf.split(X, y.iloc[:, 1])): 
            X_train, X_test = X.iloc[train_index], X.iloc[test_index]
            y_train_df, y_test_df = y.iloc[train_index], y.iloc[test_index]
            
            # y_train into array 
            y_train = [] 
            for i, j in zip(y_train_df['event_DFS'], y_train_df['DFS']): 
                y_train.append((i, j))
            y_train = np.array(y_train, dtype=[('status', bool), ('time', np.int32)])

            # y_test into array
            y_test = [] 
            for i, j in zip(y_test_df['event_DFS'], y_test_df['DFS']): 
                y_test.append((i, j))
            y_test = np.array(y_test, dtype=[('status', bool), ('time', np.int32)])

            # Standardization
            excluded_columns = ['female', 
                                'cavum_oris',
                                'oropharynx',
                                'hypopharynx',
                                'larynx',
                                'histgrade_high',
                                'hpv_related',
                                'charlson',
                                'uicc8_III-IV'
                               ]
            excluded_columns = set(excluded_columns).intersection(X.columns)

            scaler = StandardScaler() 
            X_train_included = X_train.drop(excluded_columns, axis=1)
            X_test_included = X_test.drop(excluded_columns, axis=1)
                        
            if not X_train_included.empty and not X_test_included.empty:
                X_train_included_std = scaler.fit_transform(X_train_included)
                X_test_included_std = scaler.transform(X_test_included)
                
                # Concatenation
                X_train_std_df = pd.DataFrame(X_train_included_std, columns=X_train_included.columns, index=X_train_included.index)
                X_train_std = pd.concat([X_train_std_df, X_train[excluded_columns]], axis=1)

                X_test_std_df = pd.DataFrame(X_test_included_std, columns=X_test_included.columns, index=X_test_included.index)
                X_test_std = pd.concat([X_test_std_df, X_test[excluded_columns]], axis=1)
            
            else: 
                X_train_std = X_train
                X_test_std = X_test 
            
            model.fit(X_train_std, y_train)

            if metric == "c-index":
                # Make predictions using C-index 
                c_index_score = model.score(X_test_std, y_test)
                scores.append(c_index_score)
                print(f"Fold {k + 1} C-index: {c_index_score}")
                
            elif metric == "ibs":
                # Make predictions using IBS 
                lower, upper = np.percentile(y_test["time"], [10, 90])    
                times = np.arange(lower, upper)
                surv_prob = np.row_stack([fn(times) for fn in model.predict_survival_function(X_test_std)])
                ibs = integrated_brier_score(y_test, y_test, surv_prob, times)
                scores.append(ibs)
                print(f"Fold {k + 1} IBS: {ibs}")
            else:
                raise ValueError("Invalid metric. Use 'C-index' or 'ibs'.")
        
        # Return the mean of scores
        return np.mean(scores)
    
    return objective


# C-index
study_cindex = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=123))
objective_cindex = create_objective(ComponentwiseGradientBoostingSurvivalAnalysis, "c-index", X_new, y)
study_cindex.optimize(objective_cindex, n_trials=100, show_progress_bar=True)
print("\n")
print("* Best trial for C-index: \n", study_cindex.best_trial)
print("\n")
print("* Best hyperparameters for C-index: \n", study_cindex.best_params)
print("\n")
print("* Best Score for C-index: \n", study_cindex.best_value)

# IBS
study_ibs = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=123))
objective_ibs = create_objective(ComponentwiseGradientBoostingSurvivalAnalysis, "ibs", X_new, y)
study_ibs.optimize(objective_ibs, n_trials=100, show_progress_bar=True)
print("\n")
print("* Best trial for IBS: \n", study_ibs.best_trial)
print("\n")
print("* Best hyperparameters for IBS: \n", study_ibs.best_params)
print("\n")
print("* Best Score for IBS: \n", study_ibs.best_value)


[I 2024-04-16 23:55:08,036] A new study created in memory with name: no-name-ec096de9-45ee-4dd7-a750-15600a0b2f9a


  0%|          | 0/100 [00:00<?, ?it/s]

Fold 1 C-index: 0.5657370517928287
Fold 2 C-index: 0.6976744186046512
Fold 3 C-index: 0.5617021276595745
Fold 4 C-index: 0.6387832699619772
Fold 5 C-index: 0.6609442060085837
[I 2024-04-16 23:55:10,819] Trial 0 finished with value: 0.6249682148055231 and parameters: {'subsample': 0.7268222670380755, 'dropout_rate': 0.3575254014553415, 'n_estimators': 114, 'learning_rate': 0.05558016213920623}. Best is trial 0 with value: 0.6249682148055231.
Fold 1 C-index: 0.5657370517928287
Fold 2 C-index: 0.6976744186046512
Fold 3 C-index: 0.5617021276595745
Fold 4 C-index: 0.6387832699619772
Fold 5 C-index: 0.6566523605150214
[I 2024-04-16 23:55:24,538] Trial 1 finished with value: 0.6241098457068107 and parameters: {'subsample': 0.7475220728070068, 'dropout_rate': 0.4807958141120149, 'n_estimators': 491, 'learning_rate': 0.06879814411990147}. Best is trial 0 with value: 0.6249682148055231.
Fold 1 C-index: 0.5657370517928287
Fold 2 C-index: 0.7015503875968992
Fold 3 C-index: 0.5617021276595745
Fold 

Fold 1 C-index: 0.5816733067729084
Fold 2 C-index: 0.7170542635658915
Fold 3 C-index: 0.5659574468085107
Fold 4 C-index: 0.6692015209125475
Fold 5 C-index: 0.6738197424892703
[I 2024-04-16 23:57:37,614] Trial 19 finished with value: 0.6415412561098257 and parameters: {'subsample': 0.10013306718236009, 'dropout_rate': 0.6903547530936671, 'n_estimators': 262, 'learning_rate': 0.09294018751145315}. Best is trial 14 with value: 0.6426929839942281.
Fold 1 C-index: 0.5617529880478087
Fold 2 C-index: 0.6976744186046512
Fold 3 C-index: 0.5617021276595745
Fold 4 C-index: 0.6539923954372624
Fold 5 C-index: 0.6781115879828327
[I 2024-04-16 23:57:40,107] Trial 20 finished with value: 0.6306467035464258 and parameters: {'subsample': 0.2630057481431337, 'dropout_rate': 0.7595640861978724, 'n_estimators': 112, 'learning_rate': 0.07955999934197197}. Best is trial 14 with value: 0.6426929839942281.
Fold 1 C-index: 0.5816733067729084
Fold 2 C-index: 0.7209302325581395
Fold 3 C-index: 0.5617021276595745


Fold 1 C-index: 0.5896414342629482
Fold 2 C-index: 0.7093023255813954
Fold 3 C-index: 0.5574468085106383
Fold 4 C-index: 0.6653992395437263
Fold 5 C-index: 0.6738197424892703
[I 2024-04-16 23:59:43,294] Trial 38 finished with value: 0.6391219100775957 and parameters: {'subsample': 0.14784367538324486, 'dropout_rate': 0.6936459800104628, 'n_estimators': 263, 'learning_rate': 0.07662382993812163}. Best is trial 31 with value: 0.6431600814823024.
Fold 1 C-index: 0.5617529880478087
Fold 2 C-index: 0.7054263565891473
Fold 3 C-index: 0.5617021276595745
Fold 4 C-index: 0.6501901140684411
Fold 5 C-index: 0.6695278969957081
[I 2024-04-16 23:59:47,297] Trial 39 finished with value: 0.6297198966721359 and parameters: {'subsample': 0.3156029735189495, 'dropout_rate': 0.6155875677640822, 'n_estimators': 159, 'learning_rate': 0.06699899580023713}. Best is trial 31 with value: 0.6431600814823024.
Fold 1 C-index: 0.5617529880478087
Fold 2 C-index: 0.7015503875968992
Fold 3 C-index: 0.5617021276595745


...

Fold 3 IBS: 0.2756019255493613
Fold 4 IBS: 0.24374223787259539
Fold 5 IBS: 0.22235550248109837
[I 2024-04-17 00:08:05,382] Trial 19 finished with value: 0.24322147456443471 and parameters: {'subsample': 0.3892284838807411, 'dropout_rate': 0.5354432466556798, 'n_estimators': 131, 'learning_rate': 0.023294697221931306}. Best is trial 17 with value: 0.22661013861867835.
Fold 1 IBS: 0.24550608887605468
Fold 2 IBS: 0.2133505584021398
Fold 3 IBS: 0.23294747738469943
Fold 4 IBS: 0.23024731593258208
Fold 5 IBS: 0.2123372739360029
[I 2024-04-17 00:08:06,883] Trial 20 finished with value: 0.22687774290629575 and parameters: {'subsample': 0.6210873354088753, 'dropout_rate': 0.7933084651006226, 'n_estimators': 66, 'learning_rate': 0.013007963411749002}. Best is trial 17 with value: 0.22661013861867835.
Fold 1 IBS: 0.2448259892509062
Fold 2 IBS: 0.21786672573833757
Fold 3 IBS: 0.23050391968540718
Fold 4 IBS: 0.23249100529840805
Fold 5 IBS: 0.215533050087791
[I 2024-04-17 00:08:08,802] Trial 21 fini

Fold 3 IBS: 0.2566141669263308
Fold 4 IBS: 0.23453614765450895
Fold 5 IBS: 0.2122914778844625
[I 2024-04-17 00:09:04,092] Trial 38 finished with value: 0.2333975796328732 and parameters: {'subsample': 0.605229739689187, 'dropout_rate': 0.13272164755980653, 'n_estimators': 176, 'learning_rate': 0.01283698591663533}. Best is trial 17 with value: 0.22661013861867835.
Fold 1 IBS: 0.2943463281154152
Fold 2 IBS: 0.21435275898375683
Fold 3 IBS: 0.292776623523439
Fold 4 IBS: 0.2606234117164125
Fold 5 IBS: 0.24191548716779984
[I 2024-04-17 00:09:05,417] Trial 39 finished with value: 0.26080292190136467 and parameters: {'subsample': 0.6970162917669863, 'dropout_rate': 0.48841341315505027, 'n_estimators': 59, 'learning_rate': 0.06892741183938003}. Best is trial 17 with value: 0.22661013861867835.
Fold 1 IBS: 0.2956705905294083
Fold 2 IBS: 0.21430445848659016
Fold 3 IBS: 0.2928588854990794
Fold 4 IBS: 0.2596622123064148
Fold 5 IBS: 0.24678618384472736
[I 2024-04-17 00:09:07,993] Trial 40 finished 

Fold 4 IBS: 0.28596472610806994
Fold 5 IBS: 0.27461860801929316
[I 2024-04-17 00:09:43,966] Trial 57 finished with value: 0.29238736666151666 and parameters: {'subsample': 0.763955514973389, 'dropout_rate': 0.9964559295946985, 'n_estimators': 282, 'learning_rate': 0.02203371126489022}. Best is trial 56 with value: 0.22659309077213977.
Fold 1 IBS: 0.2524958013314946
Fold 2 IBS: 0.20467957580837548
Fold 3 IBS: 0.24466604990129345
Fold 4 IBS: 0.22951388023914565
Fold 5 IBS: 0.21232254648299242
[I 2024-04-17 00:09:45,079] Trial 58 finished with value: 0.22873557075266032 and parameters: {'subsample': 0.8210665501105633, 'dropout_rate': 0.9557982366790785, 'n_estimators': 50, 'learning_rate': 0.03153701184110286}. Best is trial 56 with value: 0.22659309077213977.
Fold 1 IBS: 0.2450766728615442
Fold 2 IBS: 0.21570571438138603
Fold 3 IBS: 0.23170031253784298
Fold 4 IBS: 0.23100841297079955
Fold 5 IBS: 0.21387190006582926
[I 2024-04-17 00:09:45,627] Trial 59 finished with value: 0.227472602563

Fold 4 IBS: 0.2318272715320025
Fold 5 IBS: 0.21495710360760373
[I 2024-04-17 00:10:41,548] Trial 76 finished with value: 0.22786680538358572 and parameters: {'subsample': 0.5683268745859826, 'dropout_rate': 0.8653081987830238, 'n_estimators': 34, 'learning_rate': 0.020347121996093395}. Best is trial 56 with value: 0.22659309077213977.
Fold 1 IBS: 0.2506811440240583
Fold 2 IBS: 0.2053539697592853
Fold 3 IBS: 0.2432701864482772
Fold 4 IBS: 0.2284683602008423
Fold 5 IBS: 0.21006566848816424
[I 2024-04-17 00:10:44,958] Trial 77 finished with value: 0.22756786578412544 and parameters: {'subsample': 0.5096986308003991, 'dropout_rate': 0.49881327862855496, 'n_estimators': 145, 'learning_rate': 0.010267257888313383}. Best is trial 56 with value: 0.22659309077213977.
Fold 1 IBS: 0.24728609182637104
Fold 2 IBS: 0.20948456237788454
Fold 3 IBS: 0.2367879689804507
Fold 4 IBS: 0.22900068533758589
Fold 5 IBS: 0.21042389725248523
[I 2024-04-17 00:10:46,600] Trial 78 finished with value: 0.226596641154

Fold 4 IBS: 0.23195611702074945
Fold 5 IBS: 0.21507941572440312
[I 2024-04-17 00:11:18,411] Trial 95 finished with value: 0.22795675803795015 and parameters: {'subsample': 0.5665760978972413, 'dropout_rate': 0.4628605993177724, 'n_estimators': 100, 'learning_rate': 0.006666098315025223}. Best is trial 81 with value: 0.2265598636628418.
Fold 1 IBS: 0.3528807852002724
Fold 2 IBS: 0.26163876419301324
Fold 3 IBS: 0.35927296600690356
Fold 4 IBS: 0.2978922662575483
Fold 5 IBS: 0.30289336976611636
[I 2024-04-17 00:11:22,569] Trial 96 finished with value: 0.31491563028477076 and parameters: {'subsample': 0.41313939490511475, 'dropout_rate': 0.5213109482382285, 'n_estimators': 158, 'learning_rate': 0.054487916689644846}. Best is trial 81 with value: 0.2265598636628418.
Fold 1 IBS: 0.24708172105946397
Fold 2 IBS: 0.21005062605084635
Fold 3 IBS: 0.23658239945020892
Fold 4 IBS: 0.2291089640988238
Fold 5 IBS: 0.2105403389046395
[I 2024-04-17 00:11:24,429] Trial 97 finished with value: 0.22667280991

In [75]:
train_cindex['ComponentwiseGradientBoosting'] = np.round(study_cindex.best_value, 3)
train_ibs['ComponentwiseGradientBoosting'] = np.round(study_ibs.best_value, 3)

In [76]:
print("train_cindex: ", np.round(study_cindex.best_value, 3))
print("train_ibs: ", np.round(study_ibs.best_value, 3))

train_cindex:  0.643
train_ibs:  0.227


#### Test

In [77]:
# y into array 
lists = [] 
for i, j in zip(y['event_DFS'], y['DFS']): 
    lists.append((i, j))

y = np.array(lists, dtype=[('status', bool), ('time', np.int32)])

In [78]:
# A function for building the best model with the best parameters 
def create_best_model(model_class, best_params):
    best_params["random_state"]=123
    return model_class(**best_params)

# Set the best model 
best_model_cindex = create_best_model(ComponentwiseGradientBoostingSurvivalAnalysis, study_cindex.best_params)

# Train the best model for C-index on the whole dataset
best_model_cindex.fit(X_new_std, y)

# Evaluate the best model for C-index on MAASTRO dataset
c_index = best_model_cindex.score(MAASTRO_new_std, y_MAASTRO)
c_index = np.round(c_index, 3)
print("C-index score:", c_index)

# Set the best model 
best_model_ibs = create_best_model(ComponentwiseGradientBoostingSurvivalAnalysis, study_ibs.best_params)

# Train the best model for IBS on the whole dataset
best_model_ibs.fit(X_new_std, y)

# Evaluate the best model for IBS on MAASTRO dataset
lower, upper = np.percentile(y_MAASTRO["time"], [10, 90])
times = np.arange(lower, upper)
surv_prob = np.row_stack([fn(times) for fn in best_model_ibs.predict_survival_function(MAASTRO_new_std)])
ibs = integrated_brier_score(y_MAASTRO, y_MAASTRO, surv_prob, times)
ibs = np.round(ibs, 3)
print("IBS:", ibs)

ComponentwiseGradientBoostingSurvivalAnalysis(dropout_rate=0.5849037326403841,
                                              learning_rate=0.0943922625884125,
                                              n_estimators=305,
                                              random_state=123,
                                              subsample=0.13514815140998385)

C-index score: 0.531


ComponentwiseGradientBoostingSurvivalAnalysis(dropout_rate=0.6341528845468055,
                                              learning_rate=0.014291926646619518,
                                              n_estimators=76, random_state=123,
                                              subsample=0.6070545538128025)

IBS: 0.234


In [79]:
# Saving the values to the dictionary 
test_cindex['ComponentwiseGradientBoosting'] = c_index
test_ibs['ComponentwiseGradientBoosting'] = ibs

## Results

In [80]:
df_train_cindex = pd.DataFrame(train_cindex, index=['C-index']).transpose().sort_values(by='C-index', ascending=False)
df_train_cindex['rank'] = df_train_cindex['C-index'].rank(ascending=False)
df_train_cindex 

,C-index,rank
Randomsurvivalforest,0.847,1.0
ExtraSurvivalTrees,0.842,2.0
GradientBoosting,0.700,3.0
ComponentwiseGradientBoosting,0.643,4.0
CoxElastic,0.639,5.0
CoxRidge,0.633,6.0
CoxLasso,0.506,7.0


In [81]:
df_train_ibs = pd.DataFrame(train_ibs, index=['IBS']).transpose().sort_values(by='IBS', ascending=True)
df_train_ibs['rank'] = df_train_ibs['IBS'].rank(ascending=True)
df_train_ibs

,IBS,rank
ExtraSurvivalTrees,0.208,1.0
Randomsurvivalforest,0.212,2.0
ComponentwiseGradientBoosting,0.227,3.0
GradientBoosting,0.230,4.0
CoxElastic,0.234,5.0
CoxRidge,0.236,6.0
CoxLasso,0.432,7.0


In [82]:
df_test_cindex = pd.DataFrame(test_cindex, index=['C-index']).transpose().sort_values(by='C-index', ascending=False)
df_test_cindex['rank'] = df_test_cindex['C-index'].rank(ascending=False)
df_test_cindex 

,C-index,rank
ExtraSurvivalTrees,0.567,1.0
CoxLasso,0.545,2.0
CoxRidge,0.542,3.0
CoxElastic,0.540,4.5
Randomsurvivalforest,0.540,4.5
ComponentwiseGradientBoosting,0.531,6.0
GradientBoosting,0.516,7.0


In [83]:
df_test_ibs = pd.DataFrame(test_ibs, index=['IBS']).transpose().sort_values(by='IBS', ascending=True)
df_test_ibs['rank'] = df_test_ibs['IBS'].rank(ascending=True)
df_test_ibs

,IBS,rank
CoxElastic,0.228,1.5
GradientBoosting,0.228,1.5
CoxRidge,0.229,3.0
ComponentwiseGradientBoosting,0.234,4.0
ExtraSurvivalTrees,0.235,5.0
Randomsurvivalforest,0.262,6.0
CoxLasso,0.431,7.0


In [84]:
# Renaming the column "index" to "model" 
df_train_cindex = df_train_cindex.reset_index().rename(columns={"index": "model"})
df_train_ibs = df_train_ibs.reset_index().rename(columns={"index": "model"})
df_test_cindex = df_test_cindex.reset_index().rename(columns={"index": "model"})
df_test_ibs = df_test_ibs.reset_index().rename(columns={"index": "model"})

# Save the files 
dfs = [df_train_cindex, df_train_ibs, df_test_cindex, df_test_ibs]  # List of your DataFrames
file_path = 'path_to_your_folder/'  # Folder path where you want to save the files

# List of corresponding file names
file_names = ['train_cindex.csv', 'train_ibs.csv', 'test_cindex.csv', 'test_ibs.csv']


dfs = [df_train_cindex, df_train_ibs, df_test_cindex, df_test_ibs]  # List of your DataFrames
file_path = '/Users/minjeongcheon/Desktop/results_thesis/d3/dfs/standard/no_selection/'  # Folder path where you want to save the files

# List of corresponding file namess
file_names = ['train_cindex.csv', 'train_ibs.csv', 'test_cindex.csv', 'test_ibs.csv']

# Modify the file names to match the desired format
modified_file_names = ['d3_dfs_standard_no_selection_' + file_name for file_name in file_names]

# Loop through each DataFrame and save them with corresponding modified file names
for df, modified_file_name in zip(dfs, modified_file_names):
    file_path_name = file_path + modified_file_name  # Construct the full file path
    df.to_csv(file_path_name, index=False)  # Save the DataFrame to CSV file


In [85]:
from datetime import date
today = date.today()
print("Date: ", today)

Date:  2024-04-17
